<a href="https://colab.research.google.com/github/Kathy42xu/DL_TA/blob/main/ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#data extraction


In [ ]:
import pandas as pd
import os
import time

data_folder_path = './'

num_files = 10
file_prefix = 'factor_'
file_suffix = '.csv'

all_data_parts = []
column_names = None

print(f"--- Starting Data Combination ---")
start_load_time = time.time()
print(f"Attempting to load files from {data_folder_path}...")
print("Expecting header ONLY in factor_1.csv")

if not os.path.isdir(data_folder_path):
    print(f"ERROR: Directory not found: {data_folder_path}")
    print("Please ensure you have uploaded the 'data' folder containing the CSV files.")
    # Handle error appropriately, maybe raise exception or exit
    combined_data = None # Set combined_data to None or handle error
else:
    # --- Load the first file (factor_1.csv) WITH header ---
    file_name_1 = f"{file_prefix}1{file_suffix}"
    file_path_1 = os.path.join(data_folder_path, file_name_1)

    if os.path.exists(file_path_1):
        print(f"  Loading {file_path_1} (with header)...")
        try:
            df_first = pd.read_csv(file_path_1)
            column_names = df_first.columns.tolist() # Store column names
            all_data_parts.append(df_first)
            print(f"    Successfully loaded {file_name_1}. Shape: {df_first.shape}")
            if column_names:
                 print(f"    Stored {len(column_names)} column names.")
            else:
                 print("    Warning: No column names found in header file.")
                 column_names = None # Treat as failure if no columns read

        except Exception as e:
            print(f"    ERROR loading header file {file_name_1}: {e}")
            column_names = None # Ensure it's None if loading failed
    else:
        print(f"  ERROR: Header file not found - {file_path_1}. Cannot proceed.")
        column_names = None # Ensure it's None if file not found

    # --- Load subsequent files (factor_2.csv to factor_10.csv) WITHOUT header ---
    if column_names is not None: # Proceed only if factor_1 was loaded successfully
        for i in range(2, num_files + 1):
            file_name = f"{file_prefix}{i}{file_suffix}"
            file_path = os.path.join(data_folder_path, file_name)

            if os.path.exists(file_path):
                print(f"  Loading {file_path} (WITHOUT header)...")
                try:
                    # Load WITHOUT header
                    df_temp = pd.read_csv(file_path, header=None)

                    # Important Check: Verify column count matches factor_1
                    if len(df_temp.columns) != len(column_names):
                         print(f"    WARNING: {file_name} has {len(df_temp.columns)} columns, but header has {len(column_names)}. Skipping this file.")
                         continue # Skip this file

                    # Assign correct column names HERE
                    df_temp.columns = column_names

                    # Add the DataFrame (now with correct names) to the list
                    all_data_parts.append(df_temp)
                    # print(f"    Successfully loaded {file_name}. Shape: {df_temp.shape}") # Reduced verbosity
                except Exception as e:
                    print(f"    ERROR loading {file_name}: {e}")
                    # continue # Optional: skip failing files
            else:
                print(f"  WARNING: File not found - {file_path}. Skipping.")

    # --- Concatenate ---
    combined_data = None # Initialize in case of errors
    if not all_data_parts:
        print("\nERROR: No data parts were loaded. Cannot concatenate.")
    elif column_names is None:
         print("\nERROR: Could not load header from factor_1.csv. Cannot proceed.")
    else:
        print("\nConcatenating all loaded data parts...")
        # Concatenate DataFrames. All parts should now have correct column names.
        combined_data = pd.concat(all_data_parts, ignore_index=True)
        print("Concatenation complete!")

        # Final check on columns (should pass now)
        if len(combined_data.columns) != len(column_names):
             print("ERROR: Final column count mismatch after assigning names. Check logic.")
             combined_data = None # Invalidate data if something went wrong
        else:
             # Display the shape and head/tail of the final combined DataFrame
             print("\nShape of the final combined DataFrame:")
             print(combined_data.shape)
             print("\nFirst 5 rows:")
             print(combined_data.head())
             print("\nLast 5 rows:")
             print(combined_data.tail())

# --- End of Data Combination Cell ---
load_end_time = time.time()
print(f"\n--- Data Combination Finished ---")
if combined_data is not None:
    print(f"Variable 'combined_data' created. Shape: {combined_data.shape}")
    print(f"Time taken: {load_end_time - start_load_time:.2f} seconds")
else:
    print("Variable 'combined_data' could not be created due to errors.")

#OLS-3+H Model (Huber Loss)

In [ ]:

import pandas as pd
import numpy as np
import os
from sklearn.linear_model import HuberRegressor # Using Huber for robust regression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings
import time

warnings.filterwarnings("ignore")

# --- Configuration ---
# (Keep the same configuration as before: data_folder_path, feature_cols, target_col, date ranges etc.)
data_folder_path = './'
num_files = 10
file_prefix = 'factor_'
file_suffix = '.csv'

feature_cols = [
    'size_rnk',
    'BPR',
    'mom12'
]
target_col = 'target'
date_col = 'date'
ticker_col = 'ticker'

train_start_year = 2002
# train_end_year = 2015 # Determined dynamically
test_start_year = 2019
test_end_year = 2020


# --- Functions ---
def preprocess_combined_data(df, date_col, ticker_col, target_col, feature_cols):
    """Preprocesses the combined dataframe with numeric conversion and median+0 imputation for features."""
    print("--- Starting Data Preprocessing ---")
    if df is None:
        print("ERROR: Input DataFrame is None.")
        return None
    if not isinstance(df, pd.DataFrame):
        print("ERROR: Input is not a pandas DataFrame.")
        return None

    df_processed = df.copy()
    print(f"Input shape: {df_processed.shape}")

    # Convert date column if it's not the index already
    if date_col in df_processed.columns:
        print(f"Converting '{date_col}' to datetime...")
        df_processed[date_col] = pd.to_datetime(df_processed[date_col], errors='coerce')
        df_processed = df_processed.dropna(subset=[date_col])
        df_processed = df_processed.set_index(date_col)
    elif not isinstance(df_processed.index, pd.DatetimeIndex):
        print(f"ERROR: DataFrame index is not a DatetimeIndex and '{date_col}' column not found.")
        return None
    print("Date index set.")

    # Check for required columns AFTER potential date conversion drops
    all_cols_needed = [ticker_col, target_col] + feature_cols
    missing_cols = [col for col in all_cols_needed if col not in df_processed.columns]
    if missing_cols:
        print(f"ERROR: Missing required columns: {missing_cols}")
        return None

    # Select necessary columns AFTER setting index
    df_processed = df_processed[[ticker_col, target_col] + feature_cols]

    # --- Force Feature Columns to Numeric (Convert non-numeric to NaN) ---
    print("Attempting to convert feature columns to numeric...")
    coerced_count = 0
    for col in feature_cols:
         if col in df_processed.columns:
              initial_non_numeric = pd.to_numeric(df_processed[col], errors='coerce').isna().sum() - df_processed[col].isna().sum()
              if initial_non_numeric > 0:
                   print(f"  Found {initial_non_numeric} non-numeric value(s) in '{col}'. Coercing to NaN.")
                   coerced_count += initial_non_numeric
              df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')
         else:
              print(f"Warning: Feature column {col} not found during numeric conversion.")
    if coerced_count > 0:
         print(f"Coerced a total of {coerced_count} non-numeric entries to NaN across features.")

    # Handle infinite values
    df_processed.replace([np.inf, -np.inf], np.nan, inplace=True)

    # --- Handle TARGET Variable Missing Values ---
    initial_rows = len(df_processed)
    n_target_na_initial = df_processed[target_col].isna().sum()
    if n_target_na_initial > 0:
         print(f"Found {n_target_na_initial} missing values in '{target_col}'. Dropping rows...")
    df_processed.dropna(subset=[target_col], inplace=True)
    rows_after_target_drop = len(df_processed)
    rows_dropped = initial_rows - rows_after_target_drop
    if rows_dropped > 0:
         print(f"Dropped {rows_dropped} rows due to missing '{target_col}'.")
    print(f"Shape after handling target NaNs: {df_processed.shape}")
    if rows_after_target_drop == 0:
         print("ERROR: All rows dropped due to missing target. Cannot proceed.")
         return None

    # --- Handle FEATURE Missing Values (Median then 0 Imputation) ---
    # Now includes NaNs created by coercing strings
    print("Applying Median+0 imputation to features (including coerced NaNs)...")
    df_processed['month_period'] = df_processed.index.to_period('M')
    imputed_count_zero = 0
    for col in feature_cols:
        if col in df_processed.columns:
            n_initial_na = df_processed[col].isna().sum()
            if n_initial_na > 0:
                 # print(f"  Processing feature '{col}': Found {n_initial_na} NaNs.")
                 # Use transform for median imputation
                 df_processed[col] = df_processed.groupby('month_period')[col].transform(lambda x: x.fillna(x.median()))
                 n_remaining_na = df_processed[col].isna().sum()
                 if n_remaining_na > 0:
                      df_processed[col].fillna(0, inplace=True)
                      # print(f"    Filled {n_remaining_na} remaining NAs with 0 in '{col}'.")
                      imputed_count_zero += n_remaining_na
        else:
             print(f"Warning: Feature column {col} not found during imputation.")
    df_processed.drop(columns=['month_period'], inplace=True)
    print(f"Imputation complete. Filled {imputed_count_zero} values with 0 after median imputation.")

    # Final check for NaNs
    nans_in_features = df_processed[feature_cols].isna().sum().sum()
    if nans_in_features > 0:
         print(f"WARNING: {nans_in_features} NaNs still remain in feature columns after imputation!")

    print(f"Preprocessing complete. Final shape: {df_processed.shape}")
    if not df_processed.empty:
        print(f"Final data date range: {df_processed.index.min().strftime('%Y-%m-%d')} to {df_processed.index.max().strftime('%Y-%m-%d')}")
    return df_processed

# --- R-squared Calculation Functions ---
def calculate_r2_oos(y_true, y_pred):
    """
    Calculates Out-of-Sample R-squared using the definition from the paper:
    R2_OOS = 1 - sum( (y_true - y_pred)^2 ) / sum( y_true^2 )
    This benchmarks against a naive forecast of zero.
    """
    numerator = ((y_true - y_pred) ** 2).sum()
    denominator = (y_true ** 2).sum()
    # Handle cases where the sum of squared true values is zero or very close to zero
    if np.isclose(denominator, 0):
        # If numerator is also zero, perfect fit (or all zeros); otherwise, undefined or -inf
        return 1.0 if np.isclose(numerator, 0) else -np.inf
    return 1 - (numerator / denominator)

def calculate_r2_is(y_true, y_pred):
    """
    Calculates In-Sample R-squared using the *same* formula structure as R2_OOS
    for consistency in this specific replication context.
    R2_IS = 1 - sum( (y_train_true - y_train_pred)^2 ) / sum( y_train_true^2 )
    """
    # This uses the same logic as calculate_r2_oos but is applied to training data
    return calculate_r2_oos(y_true, y_pred)


# --- Main Execution ---
print("\n--- Starting OLS-3+H Model Fitting (Huber Loss) ---")
start_time = time.time()

# Ensure the preprocessed data DataFrame 'data' exists from the previous cell run
# If 'combined_data' was the input to preprocessing in Cell 2, check for that first.
if 'combined_data' not in locals() and 'combined_data' not in globals():
     print("ERROR: 'combined_data' DataFrame not found. Please run the data combination cell first.")
     data = None
elif 'data' not in locals() and 'data' not in globals():
     # If 'data' doesn't exist, try to run preprocessing again
     print("Preprocessed 'data' not found, attempting preprocessing...")
     if 'combined_data' in locals() or 'combined_data' in globals():
         data = preprocess_combined_data(combined_data, date_col, ticker_col, target_col, feature_cols)
     else:
         data = None # Can't preprocess if combined_data is missing
elif data is None or data.empty:
    print("ERROR: Preprocessed DataFrame 'data' is None or empty.")
    # No need to set huber_results = None here, it's handled later


# Proceed with model fitting only if preprocessing was successful
if data is not None and not data.empty:
    all_oos_predictions = []
    all_oos_true_values = []
    all_is_predictions = []  # To store in-sample predictions for overall IS R2
    all_is_true_values = []   # To store in-sample true values for overall IS R2

    test_year_r2_oos = {}     # R2_OOS per test year
    train_year_r2_is = {}    # R2_IS per train period ending year

    available_years = sorted(data.index.year.unique())
    actual_test_years = [y for y in available_years if y >= test_start_year and y <= test_end_year]

    if not actual_test_years:
        print(f"\nERROR: No data available in the specified test period ({test_start_year}-{test_end_year}) after preprocessing. Cannot proceed.")
        huber_results = None
    else:
        print(f"\nStarting OLS-3+H annual refitting from {min(actual_test_years)} to {max(actual_test_years)}...")
        print(f"Using Training Start Year: {train_start_year}")

        for current_test_year in actual_test_years:
            loop_start_time = time.time()
            print(f"\n  Processing test year: {current_test_year}")

            current_train_end_year = current_test_year - 1
            if current_train_end_year < train_start_year:
                 print(f"    Skipping year {current_test_year}: Training end year ({current_train_end_year}) is before training start year ({train_start_year}).")
                 continue

            print(f"    Training period: {train_start_year}-{current_train_end_year}")
            train_mask = (data.index.year >= train_start_year) & (data.index.year <= current_train_end_year)
            test_mask = (data.index.year == current_test_year)
            train_df = data.loc[train_mask]
            test_df = data.loc[test_mask]

            if train_df.empty or test_df.empty:
                print(f"    Skipping year {current_test_year}: Not enough data for train ({len(train_df)}) / test ({len(test_df)}) split.")
                continue

            # Final check for NaNs before scaling (should be handled by preprocessing)
            if train_df[feature_cols].isna().any().any() or test_df[feature_cols].isna().any().any():
                 print(f"    ERROR: NaNs detected in features for year {current_test_year} before scaling, after preprocessing! Skipping.")
                 continue

            # Prepare data for scikit-learn
            X_train = train_df[feature_cols].values
            y_train = train_df[target_col].values
            X_test = test_df[feature_cols].values
            y_test = test_df[target_col].values

            # Scale features
            scaler = StandardScaler()
            try:
                X_train_scaled = scaler.fit_transform(X_train)
                X_test_scaled = scaler.transform(X_test)

                 # Check for NaNs again AFTER scaling (can happen with constant cols)
                if np.isnan(X_train_scaled).any() or np.isnan(X_test_scaled).any():
                    print(f"    Warning: NaNs generated during scaling for year {current_test_year}. Check for constant columns in training data. Skipping year.")
                    # Optional: Impute NaNs caused by scaling constant columns if necessary
                    # X_train_scaled = np.nan_to_num(X_train_scaled)
                    # X_test_scaled = np.nan_to_num(X_test_scaled)
                    continue # Skip this year if NaNs appear after scaling

            except ValueError as e:
                print(f"    ERROR during scaling for year {current_test_year}: {e}. Skipping year.")
                continue

            # --- Huber Regressor Model Training and Prediction ---
            # Using HuberRegressor for robustness to outliers, as in the original cell
            huber_model = HuberRegressor(fit_intercept=True, max_iter=300, tol=1e-4)

            try:
                # Check for NaNs before fitting/predicting
                if np.isnan(X_train_scaled).any() or np.isnan(y_train).any():
                     print(f"    ERROR: NaNs detected in training data before fitting model for year {current_test_year}. Skipping.")
                     continue
                if np.isnan(X_test_scaled).any():
                     print(f"    ERROR: NaNs detected in test features before prediction for year {current_test_year}. Skipping.")
                     continue

                # Fit the Huber model
                huber_model.fit(X_train_scaled, y_train)

                # --- In-Sample Prediction ---
                is_predictions = huber_model.predict(X_train_scaled)
                all_is_predictions.extend(is_predictions)
                all_is_true_values.extend(y_train)
                r2_is_this_year = calculate_r2_is(y_train, is_predictions)
                train_year_r2_is[current_train_end_year] = r2_is_this_year # Store IS R2 by training end year

                # --- Out-of-Sample Prediction ---
                oos_predictions = huber_model.predict(X_test_scaled)
                all_oos_predictions.extend(oos_predictions)
                all_oos_true_values.extend(y_test)
                r2_oos_this_year = calculate_r2_oos(y_test, oos_predictions)
                test_year_r2_oos[current_test_year] = r2_oos_this_year # Store OOS R2 by test year

                print(f"    Train Year {current_train_end_year} R2_IS : {r2_is_this_year:+.4f}")
                print(f"    Test Year  {current_test_year} R2_OOS: {r2_oos_this_year:+.4f}")
                print(f"    Time for year: {time.time() - loop_start_time:.2f}s")

            except Exception as e:
                print(f"    ERROR during model fitting/prediction for year {current_test_year}: {e}")


        # --- Overall Results ---
        print("\n--- Overall OLS-3+H Results (Huber Loss) ---")

        # In-Sample Overall
        overall_r2_is = None # Initialize
        if all_is_true_values:
            overall_r2_is = calculate_r2_is(np.array(all_is_true_values), np.array(all_is_predictions))
            print(f"\nOverall In-Sample R-squared (R2_IS) across all training periods: {overall_r2_is:.4f}")
            print("\nR2_IS per training period (end year):")
            for year, r2 in sorted(train_year_r2_is.items()):
                print(f"  {train_start_year}-{year}: {r2:.4f}")
        else:
            print("\nNo valid in-sample predictions were generated.")

        # Out-of-Sample Overall
        overall_r2_oos = None # Initialize
        if all_oos_true_values:
            overall_r2_oos = calculate_r2_oos(np.array(all_oos_true_values), np.array(all_oos_predictions))
            print(f"\nOverall Out-of-Sample R-squared (R2_OOS) for period {min(actual_test_years)}-{max(actual_test_years)}: {overall_r2_oos:.4f}")
            # Compare to OLS-3 benchmark from paper if desired
            print(f"(Paper's OLS-3 benchmark R2_OOS: 0.16% - Note: Direct comparison depends on exact data/setup)")
            print("\nR2_OOS per test year:")
            for year, r2 in sorted(test_year_r2_oos.items()):
                print(f"  {year}: {r2:.4f}")
        else:
            print("\nNo valid out-of-sample predictions were generated for the test period.")

        # Store results in a dictionary
        huber_results = {
            "overall_r2_is": overall_r2_is,
            "overall_r2_oos": overall_r2_oos,
            "yearly_r2_is": train_year_r2_is,
            "yearly_r2_oos": test_year_r2_oos
        }

else:
    # This block executes if data is None or empty after preprocessing attempt
    print("\nData preprocessing failed or resulted in empty DataFrame. Cannot proceed with model fitting.")
    huber_results = None # Indicate failure

# Final summary
end_time = time.time()
print(f"\nTotal execution time for OLS-3+H cell: {end_time - start_time:.2f} seconds")

# Check if results were successfully generated
if 'huber_results' in locals() and huber_results is not None:
    print("\nOLS-3+H model fitting complete. Results stored in 'huber_results' dictionary.")
elif 'huber_results' not in locals() or huber_results is None:
     print("\nOLS-3+H model fitting did not complete successfully due to errors or lack of data.")



#OLS-All+H Model (Huber Loss, All Features)

In [ ]:

import pandas as pd
import numpy as np
import os
from sklearn.linear_model import HuberRegressor # Using Huber for robust regression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings
import time

warnings.filterwarnings("ignore")

# --- Configuration ---
# (Keep the same configuration as before: data_folder_path, num_files etc.)
data_folder_path = './'
num_files = 10
file_prefix = 'factor_'
file_suffix = '.csv'

# feature_cols will be determined dynamically later
# feature_cols = [...] # No longer hardcoded
target_col = 'target'
date_col = 'date'
ticker_col = 'ticker'

train_start_year = 2002
# train_end_year = 2015 # Determined dynamically
test_start_year = 2019
test_end_year = 2020


# --- Functions ---
def preprocess_combined_data(df, date_col, ticker_col, target_col):
    """
    Preprocesses the combined dataframe:
    - Sets date index.
    - Converts potential feature columns to numeric (coercing errors to NaN).
    - Handles infinite values.
    - Drops rows with missing target values.
    - Imputes missing feature values using monthly median, then 0.
    - Returns the processed DataFrame and the list of identified feature columns.
    """
    print("--- Starting Data Preprocessing ---")
    if df is None:
        print("ERROR: Input DataFrame is None.")
        return None, None
    if not isinstance(df, pd.DataFrame):
        print("ERROR: Input is not a pandas DataFrame.")
        return None, None

    df_processed = df.copy()
    print(f"Input shape: {df_processed.shape}")

    # 1. Convert date column and set index
    if date_col in df_processed.columns:
        print(f"Converting '{date_col}' to datetime...")
        df_processed[date_col] = pd.to_datetime(df_processed[date_col], errors='coerce')
        df_processed = df_processed.dropna(subset=[date_col]) # Drop rows where date conversion failed
        df_processed = df_processed.set_index(date_col)
        print(f"Shape after date processing: {df_processed.shape}")
    elif not isinstance(df_processed.index, pd.DatetimeIndex):
        print(f"ERROR: DataFrame index is not a DatetimeIndex and '{date_col}' column not found.")
        return None, None
    print("Date index set.")

    # 2. Identify potential feature columns (all columns except target and ticker)
    potential_feature_cols = [col for col in df_processed.columns if col not in [target_col, ticker_col]]
    print(f"Identified {len(potential_feature_cols)} potential feature columns.")
    if not potential_feature_cols:
        print("ERROR: No potential feature columns found.")
        return None, None

    # Keep only necessary columns for now (target, ticker, potential features)
    cols_to_keep = [ticker_col, target_col] + potential_feature_cols
    missing_cols = [col for col in cols_to_keep if col not in df_processed.columns]
    if missing_cols:
       # This check might be redundant if potential_feature_cols logic is correct, but good practice
        print(f"ERROR: Missing required columns after date processing: {missing_cols}")
        return None, None
    df_processed = df_processed[cols_to_keep]
    print(f"Shape after selecting columns: {df_processed.shape}")


    # 3. Force Feature Columns to Numeric (Convert non-numeric to NaN)
    print("Attempting to convert identified feature columns to numeric...")
    coerced_count = 0
    for col in potential_feature_cols:
         # Check if column still exists (it should, based on cols_to_keep)
         if col in df_processed.columns:
              # Check if column is not already numeric
              if not pd.api.types.is_numeric_dtype(df_processed[col]):
                  initial_non_numeric = pd.to_numeric(df_processed[col], errors='coerce').isna().sum() - df_processed[col].isna().sum()
                  if initial_non_numeric > 0:
                       print(f"  Found {initial_non_numeric} non-numeric value(s) in '{col}'. Coercing to NaN.")
                       coerced_count += initial_non_numeric
                  df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')
              #else:
              #     print(f" Column '{col}' is already numeric.") # Optional: for debugging
         else:
              # This case should ideally not happen if cols_to_keep logic is correct
              print(f"Warning: Potential feature column '{col}' not found during numeric conversion phase.")

    if coerced_count > 0:
         print(f"Coerced a total of {coerced_count} non-numeric entries to NaN across features.")

    # 4. Handle infinite values
    df_processed.replace([np.inf, -np.inf], np.nan, inplace=True)
    print("Handled infinite values.")

    # 5. Handle TARGET Variable Missing Values
    initial_rows = len(df_processed)
    n_target_na_initial = df_processed[target_col].isna().sum()
    if n_target_na_initial > 0:
         print(f"Found {n_target_na_initial} missing values in '{target_col}'. Dropping rows...")
         df_processed.dropna(subset=[target_col], inplace=True)
         rows_after_target_drop = len(df_processed)
         rows_dropped = initial_rows - rows_after_target_drop
         if rows_dropped > 0:
              print(f"Dropped {rows_dropped} rows due to missing '{target_col}'.")
    else:
        print(f"No missing values found in '{target_col}'.")

    print(f"Shape after handling target NaNs: {df_processed.shape}")
    if df_processed.empty:
         print("ERROR: All rows dropped due to missing target or date issues. Cannot proceed.")
         return None, None

    # --- Handle FEATURE Missing Values (Median then 0 Imputation) ---
    # Now includes NaNs created by coercing strings or from original data
    print("Applying Median+0 imputation to features...")
    # Calculate monthly periods for grouping
    try:
        df_processed['month_period'] = df_processed.index.to_period('M')
    except AttributeError as e:
        print(f"Error creating 'month_period': {e}. Ensure index is DatetimeIndex.")
        return None, None

    imputed_count_median = 0
    imputed_count_zero = 0
    final_feature_cols = [] # Store columns that are actually used after checks

    for col in potential_feature_cols:
        if col in df_processed.columns:
            n_initial_na = df_processed[col].isna().sum()
            if n_initial_na > 0:
                # print(f"  Processing feature '{col}': Found {n_initial_na} NaNs.") # Verbose
                # Group by month and impute median within each month
                try:
                    # Calculate medians per group, handling potential all-NaN groups
                    medians = df_processed.groupby('month_period')[col].median()
                    # Fill NaNs using transform
                    df_processed[col] = df_processed.groupby('month_period')[col].transform(lambda x: x.fillna(medians.loc[x.name]))
                    imputed_this_median = n_initial_na - df_processed[col].isna().sum()
                    imputed_count_median += imputed_this_median
                except Exception as e:
                    print(f"    WARNING: Could not impute median for '{col}' using groupby/transform. Error: {e}. Check data for this column.")
                    # Fallback or skip? For now, we'll let the fillna(0) handle remaining NaNs.

                # Impute remaining NaNs (if median was NaN for a whole group) with 0
                n_remaining_na = df_processed[col].isna().sum()
                if n_remaining_na > 0:
                    df_processed[col].fillna(0, inplace=True)
                    # print(f"    Filled {n_remaining_na} remaining NAs with 0 in '{col}'.") # Verbose
                    imputed_count_zero += n_remaining_na
            # Check if column is numeric after potential imputation
            if pd.api.types.is_numeric_dtype(df_processed[col]):
                 final_feature_cols.append(col)
            else:
                 print(f"  WARNING: Column '{col}' is not numeric after imputation. Excluding from final features.")
        else:
             # Should not happen
             print(f"Warning: Potential feature column '{col}' not found during imputation phase.")

    df_processed.drop(columns=['month_period'], inplace=True) # Remove temporary column
    print(f"Imputation complete. Imputed ~{imputed_count_median} values with median, {imputed_count_zero} values with 0.")

    # Final check for NaNs in the final feature set
    if not final_feature_cols:
        print("ERROR: No numeric feature columns remain after preprocessing.")
        return None, None

    nans_in_features = df_processed[final_feature_cols].isna().sum().sum()
    if nans_in_features > 0:
         print(f"WARNING: {nans_in_features} NaNs still remain in final feature columns after imputation! This should not happen.")
         # Consider dropping columns/rows with remaining NaNs or using a different imputer
         # For now, we will proceed, but the model might fail.

    print(f"Preprocessing complete. Final shape: {df_processed.shape}")
    print(f"Using {len(final_feature_cols)} features for modeling.")
    if not df_processed.empty:
        print(f"Final data date range: {df_processed.index.min().strftime('%Y-%m-%d')} to {df_processed.index.max().strftime('%Y-%m-%d')}")

    # Return only the columns needed for modeling (features + target) plus the ticker
    # Also return the identified list of feature columns
    return df_processed[[ticker_col, target_col] + final_feature_cols], final_feature_cols


# --- R-squared Calculation Functions ---
# (Keep R2 functions as they are)
def calculate_r2_oos(y_true, y_pred):
    """
    Calculates Out-of-Sample R-squared using the definition from the paper:
    R2_OOS = 1 - sum( (y_true - y_pred)^2 ) / sum( y_true^2 )
    This benchmarks against a naive forecast of zero.
    """
    numerator = ((y_true - y_pred) ** 2).sum()
    denominator = (y_true ** 2).sum()
    if np.isclose(denominator, 0):
        return 1.0 if np.isclose(numerator, 0) else -np.inf
    return 1 - (numerator / denominator)

def calculate_r2_is(y_true, y_pred):
    """
    Calculates In-Sample R-squared using the *same* formula structure as R2_OOS
    for consistency in this specific replication context.
    R2_IS = 1 - sum( (y_train_true - y_train_pred)^2 ) / sum( y_train_true^2 )
    """
    return calculate_r2_oos(y_true, y_pred)


# --- Main Execution ---
print("\n--- Starting OLS-All+H Model Fitting (Huber Loss, All Features) ---")
start_time = time.time()

# 1. Load or ensure 'combined_data' exists
if 'combined_data' not in locals() and 'combined_data' not in globals():
     print("ERROR: 'combined_data' DataFrame not found. Please run the data combination cell first.")
     data_processed = None
     feature_cols = None
else:
    # 2. Preprocess the data AND get the list of features used
    # Pass only target_col and ticker_col, let the function find features
    data_processed, feature_cols = preprocess_combined_data(combined_data, date_col, ticker_col, target_col)

# 3. Proceed with model fitting if preprocessing was successful
if data_processed is not None and not data_processed.empty and feature_cols:
    print(f"\nProceeding with {len(feature_cols)} features.")
    all_oos_predictions = []
    all_oos_true_values = []
    all_is_predictions = []
    all_is_true_values = []

    test_year_r2_oos = {}
    train_year_r2_is = {}

    available_years = sorted(data_processed.index.year.unique())
    actual_test_years = [y for y in available_years if y >= test_start_year and y <= test_end_year]

    if not actual_test_years:
        print(f"\nERROR: No data available in the specified test period ({test_start_year}-{test_end_year}) after preprocessing. Cannot proceed.")
        huber_results_all = None
    else:
        print(f"\nStarting OLS-All+H annual refitting from {min(actual_test_years)} to {max(actual_test_years)}...")
        print(f"Using Training Start Year: {train_start_year}")

        for current_test_year in actual_test_years:
            loop_start_time = time.time()
            print(f"\n  Processing test year: {current_test_year}")

            current_train_end_year = current_test_year - 1
            if current_train_end_year < train_start_year:
                 print(f"    Skipping year {current_test_year}: Training end year ({current_train_end_year}) is before training start year ({train_start_year}).")
                 continue

            print(f"    Training period: {train_start_year}-{current_train_end_year}")
            train_mask = (data_processed.index.year >= train_start_year) & (data_processed.index.year <= current_train_end_year)
            test_mask = (data_processed.index.year == current_test_year)
            train_df = data_processed.loc[train_mask]
            test_df = data_processed.loc[test_mask]

            if train_df.empty or test_df.empty:
                print(f"    Skipping year {current_test_year}: Not enough data for train ({len(train_df)}) / test ({len(test_df)}) split.")
                continue

            # Check for NaNs in the dynamically determined feature columns before scaling
            if train_df[feature_cols].isna().any().any() or test_df[feature_cols].isna().any().any():
                 print(f"    ERROR: NaNs detected in feature columns for year {current_test_year} before scaling! Skipping.")
                 # You might want to investigate which columns have NaNs here
                 # print(train_df[feature_cols].isna().sum().sort_values(ascending=False).head())
                 continue

            # Prepare data for scikit-learn using the dynamic feature_cols
            X_train = train_df[feature_cols].values
            y_train = train_df[target_col].values
            X_test = test_df[feature_cols].values
            y_test = test_df[target_col].values

            # Scale features
            scaler = StandardScaler()
            try:
                X_train_scaled = scaler.fit_transform(X_train)
                X_test_scaled = scaler.transform(X_test)

                # Check for NaNs again AFTER scaling
                if np.isnan(X_train_scaled).any() or np.isnan(X_test_scaled).any():
                    print(f"    Warning: NaNs generated during scaling for year {current_test_year}. Check for constant columns. Imputing with 0.")
                    # Impute NaNs caused by scaling constant columns with 0
                    X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
                    X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)
                    # continue # Decide whether to skip or impute

            except ValueError as e:
                print(f"    ERROR during scaling for year {current_test_year}: {e}. Skipping year.")
                continue

            # --- Huber Regressor Model Training and Prediction ---
            huber_model = HuberRegressor(fit_intercept=True, max_iter=300, tol=1e-4) # Keep max_iter reasonable

            try:
                # Final check for NaNs before fitting/predicting
                if np.isnan(X_train_scaled).any() or np.isnan(y_train).any():
                     print(f"    ERROR: NaNs detected in training data before fitting model for year {current_test_year}. Skipping.")
                     continue
                if np.isnan(X_test_scaled).any():
                     print(f"    ERROR: NaNs detected in test features before prediction for year {current_test_year}. Skipping.")
                     continue

                # Fit the Huber model
                huber_model.fit(X_train_scaled, y_train)

                # --- In-Sample Prediction ---
                is_predictions = huber_model.predict(X_train_scaled)
                all_is_predictions.extend(is_predictions)
                all_is_true_values.extend(y_train)
                r2_is_this_year = calculate_r2_is(y_train, is_predictions)
                train_year_r2_is[current_train_end_year] = r2_is_this_year

                # --- Out-of-Sample Prediction ---
                oos_predictions = huber_model.predict(X_test_scaled)
                all_oos_predictions.extend(oos_predictions)
                all_oos_true_values.extend(y_test)
                r2_oos_this_year = calculate_r2_oos(y_test, oos_predictions)
                test_year_r2_oos[current_test_year] = r2_oos_this_year

                print(f"    Train Year {current_train_end_year} R2_IS : {r2_is_this_year:+.4f}")
                print(f"    Test Year  {current_test_year} R2_OOS: {r2_oos_this_year:+.4f}")
                print(f"    Time for year: {time.time() - loop_start_time:.2f}s")

            except Exception as e:
                # Catch potential memory errors with many features
                print(f"    ERROR during model fitting/prediction for year {current_test_year}: {e}")
                if "MemoryError" in str(e):
                    print("      >>> Potential MemoryError likely due to large number of features. Consider reducing features or using a different model.")
                    # Optionally break the loop if memory errors persist
                    # break


        # --- Overall Results ---
        print("\n--- Overall OLS-All+H Results (Huber Loss, All Features) ---")

        # In-Sample Overall
        overall_r2_is = None
        if all_is_true_values:
            overall_r2_is = calculate_r2_is(np.array(all_is_true_values), np.array(all_is_predictions))
            print(f"\nOverall In-Sample R-squared (R2_IS) across all training periods: {overall_r2_is:.4f}")
            print("\nR2_IS per training period (end year):")
            for year, r2 in sorted(train_year_r2_is.items()):
                print(f"  {train_start_year}-{year}: {r2:.4f}")
        else:
            print("\nNo valid in-sample predictions were generated.")

        # Out-of-Sample Overall
        overall_r2_oos = None
        if all_oos_true_values:
            overall_r2_oos = calculate_r2_oos(np.array(all_oos_true_values), np.array(all_oos_predictions))
            print(f"\nOverall Out-of-Sample R-squared (R2_OOS) for period {min(actual_test_years)}-{max(actual_test_years)}: {overall_r2_oos:.4f}")
            print(f"(Paper's OLS-3 benchmark R2_OOS: 0.16%)") # Keep for reference
            print(f"(Paper's ENet+H (all features) R2_OOS: 0.11%)") # Add ENet comparison [source: 91]
            print("\nR2_OOS per test year:")
            for year, r2 in sorted(test_year_r2_oos.items()):
                print(f"  {year}: {r2:.4f}")
        else:
            print("\nNo valid out-of-sample predictions were generated for the test period.")

        # Store results
        huber_results_all = {
            "overall_r2_is": overall_r2_is,
            "overall_r2_oos": overall_r2_oos,
            "yearly_r2_is": train_year_r2_is,
            "yearly_r2_oos": test_year_r2_oos,
            "features_used": feature_cols # Store the list of features used
        }

else:
    # This block executes if preprocessing failed or returned empty data/features
    print("\nData preprocessing failed or resulted in empty DataFrame/features. Cannot proceed with model fitting.")
    huber_results_all = None # Indicate failure

# Final summary
end_time = time.time()
print(f"\nTotal execution time for OLS-All+H cell: {end_time - start_time:.2f} seconds")

if 'huber_results_all' in locals() and huber_results_all is not None:
    print("\nOLS-All+H model fitting complete. Results stored in 'huber_results_all' dictionary.")
    print(f"Number of features used: {len(huber_results_all.get('features_used', []))}")
elif 'huber_results_all' not in locals() or huber_results_all is None:
     print("\nOLS-All+H model fitting did not complete successfully due to errors or lack of data/features.")



#Overall PCR Results (All Features, 95% Var)

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression # PCR uses OLS in the second step
from sklearn.decomposition import PCA # For Principal Component Analysis
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import warnings
import time

warnings.filterwarnings("ignore")

# --- Configuration ---
data_folder_path = './'
num_files = 10
file_prefix = 'factor_'
file_suffix = '.csv'

# feature_cols will be determined dynamically by the preprocessing function
target_col = 'target'
date_col = 'date'
ticker_col = 'ticker'

train_start_year = 2002
test_start_year = 2019
test_end_year = 2020
pca_explained_variance_threshold = 0.95 # Use components explaining 95% of variance

# --- Functions ---

# Use the same preprocessing function from the previous step (OLS-All+H)
# It identifies features, handles numeric conversion, imputation, etc.
# and returns the processed data and the list of feature columns.
def preprocess_combined_data(df, date_col, ticker_col, target_col):
    """
    Preprocesses the combined dataframe:
    - Sets date index.
    - Converts potential feature columns to numeric (coercing errors to NaN).
    - Handles infinite values.
    - Drops rows with missing target values.
    - Imputes missing feature values using monthly median, then 0.
    - Returns the processed DataFrame and the list of identified feature columns.
    """
    print("--- Starting Data Preprocessing ---")
    if df is None:
        print("ERROR: Input DataFrame is None.")
        return None, None
    if not isinstance(df, pd.DataFrame):
        print("ERROR: Input is not a pandas DataFrame.")
        return None, None

    df_processed = df.copy()
    print(f"Input shape: {df_processed.shape}")

    # 1. Convert date column and set index
    if date_col in df_processed.columns:
        print(f"Converting '{date_col}' to datetime...")
        df_processed[date_col] = pd.to_datetime(df_processed[date_col], errors='coerce')
        df_processed = df_processed.dropna(subset=[date_col]) # Drop rows where date conversion failed
        df_processed = df_processed.set_index(date_col)
        print(f"Shape after date processing: {df_processed.shape}")
    elif not isinstance(df_processed.index, pd.DatetimeIndex):
        print(f"ERROR: DataFrame index is not a DatetimeIndex and '{date_col}' column not found.")
        return None, None
    print("Date index set.")

    # 2. Identify potential feature columns (all columns except target and ticker)
    potential_feature_cols = [col for col in df_processed.columns if col not in [target_col, ticker_col]]
    print(f"Identified {len(potential_feature_cols)} potential feature columns.")
    if not potential_feature_cols:
        print("ERROR: No potential feature columns found.")
        return None, None

    # Keep only necessary columns for now (target, ticker, potential features)
    cols_to_keep = [ticker_col, target_col] + potential_feature_cols
    missing_cols = [col for col in cols_to_keep if col not in df_processed.columns]
    if missing_cols:
        print(f"ERROR: Missing required columns after date processing: {missing_cols}")
        return None, None
    df_processed = df_processed[cols_to_keep]
    print(f"Shape after selecting columns: {df_processed.shape}")


    # 3. Force Feature Columns to Numeric (Convert non-numeric to NaN)
    print("Attempting to convert identified feature columns to numeric...")
    coerced_count = 0
    numeric_cols_found = [] # Keep track of columns confirmed numeric
    for col in potential_feature_cols:
         if col in df_processed.columns:
              if not pd.api.types.is_numeric_dtype(df_processed[col]):
                  initial_non_numeric = pd.to_numeric(df_processed[col], errors='coerce').isna().sum() - df_processed[col].isna().sum()
                  if initial_non_numeric > 0:
                       print(f"  Found {initial_non_numeric} non-numeric value(s) in '{col}'. Coercing to NaN.")
                       coerced_count += initial_non_numeric
                  df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')
              # Check again if it's numeric now (or was already)
              if pd.api.types.is_numeric_dtype(df_processed[col]):
                  numeric_cols_found.append(col)
              else:
                   print(f"  WARNING: Column '{col}' could not be converted to numeric. Excluding.")
         else:
              print(f"Warning: Potential feature column '{col}' not found during numeric conversion phase.")

    potential_feature_cols = numeric_cols_found # Update the list to only include numeric ones
    if not potential_feature_cols:
        print("ERROR: No numeric feature columns identified after conversion attempt.")
        return None, None
    print(f"Confirmed {len(potential_feature_cols)} numeric feature columns.")
    if coerced_count > 0:
         print(f"Coerced a total of {coerced_count} non-numeric entries to NaN across features.")

    # 4. Handle infinite values
    df_processed.replace([np.inf, -np.inf], np.nan, inplace=True)
    print("Handled infinite values.")

    # 5. Handle TARGET Variable Missing Values
    initial_rows = len(df_processed)
    n_target_na_initial = df_processed[target_col].isna().sum()
    if n_target_na_initial > 0:
         print(f"Found {n_target_na_initial} missing values in '{target_col}'. Dropping rows...")
         df_processed.dropna(subset=[target_col], inplace=True)
         rows_after_target_drop = len(df_processed)
         rows_dropped = initial_rows - rows_after_target_drop
         if rows_dropped > 0:
              print(f"Dropped {rows_dropped} rows due to missing '{target_col}'.")
    else:
        print(f"No missing values found in '{target_col}'.")

    print(f"Shape after handling target NaNs: {df_processed.shape}")
    if df_processed.empty:
         print("ERROR: All rows dropped due to missing target or date issues. Cannot proceed.")
         return None, None

    # --- Handle FEATURE Missing Values (Median then 0 Imputation) ---
    print("Applying Median+0 imputation to numeric features...")
    try:
        df_processed['month_period'] = df_processed.index.to_period('M')
    except AttributeError as e:
        print(f"Error creating 'month_period': {e}. Ensure index is DatetimeIndex.")
        return None, None

    imputed_count_median = 0
    imputed_count_zero = 0
    final_feature_cols = []

    for col in potential_feature_cols: # Iterate through confirmed numeric columns
        if col in df_processed.columns:
            n_initial_na = df_processed[col].isna().sum()
            if n_initial_na > 0:
                try:
                    medians = df_processed.groupby('month_period')[col].median()
                    df_processed[col] = df_processed.groupby('month_period')[col].transform(lambda x: x.fillna(medians.loc[x.name]))
                    imputed_this_median = n_initial_na - df_processed[col].isna().sum()
                    imputed_count_median += imputed_this_median
                except Exception as e:
                    print(f"    WARNING: Could not impute median for '{col}'. Error: {e}.")

                n_remaining_na = df_processed[col].isna().sum()
                if n_remaining_na > 0:
                    df_processed[col].fillna(0, inplace=True)
                    imputed_count_zero += n_remaining_na
            # Add to final list if it's still present and numeric (should be)
            if col in df_processed.columns and pd.api.types.is_numeric_dtype(df_processed[col]):
                final_feature_cols.append(col)
            else:
                 print(f"  WARNING: Column '{col}' is missing or non-numeric after imputation. Excluding.")
        else:
             print(f"Warning: Numeric feature column '{col}' not found during imputation phase.")

    df_processed.drop(columns=['month_period'], inplace=True)
    print(f"Imputation complete. Imputed ~{imputed_count_median} values with median, {imputed_count_zero} values with 0.")

    if not final_feature_cols:
        print("ERROR: No numeric feature columns remain after imputation.")
        return None, None

    nans_in_features = df_processed[final_feature_cols].isna().sum().sum()
    if nans_in_features > 0:
         print(f"WARNING: {nans_in_features} NaNs still remain in final feature columns after imputation!")

    print(f"Preprocessing complete. Final shape: {df_processed.shape}")
    print(f"Using {len(final_feature_cols)} features for modeling.")
    if not df_processed.empty:
        print(f"Final data date range: {df_processed.index.min().strftime('%Y-%m-%d')} to {df_processed.index.max().strftime('%Y-%m-%d')}")

    # Return only the columns needed for modeling
    return df_processed[[ticker_col, target_col] + final_feature_cols], final_feature_cols


# --- R-squared Calculation Functions ---
def calculate_r2_oos(y_true, y_pred):
    """Calculates Out-of-Sample R-squared (vs. zero forecast)."""
    numerator = ((y_true - y_pred) ** 2).sum()
    denominator = (y_true ** 2).sum()
    if np.isclose(denominator, 0):
        return 1.0 if np.isclose(numerator, 0) else -np.inf
    return 1 - (numerator / denominator)

def calculate_r2_is(y_true, y_pred):
    """Calculates In-Sample R-squared (vs. zero forecast)."""
    return calculate_r2_oos(y_true, y_pred)


# --- Main Execution ---
print(f"\n--- Starting PCR Model Fitting (All Features, {pca_explained_variance_threshold*100:.0f}% Variance) ---")
start_time = time.time()

# 1. Load or ensure 'combined_data' exists
if 'combined_data' not in locals() and 'combined_data' not in globals():
     print("ERROR: 'combined_data' DataFrame not found. Please run the data combination cell first.")
     data_processed = None
     feature_cols = None
else:
    # 2. Preprocess the data AND get the list of features used
    data_processed, feature_cols = preprocess_combined_data(combined_data, date_col, ticker_col, target_col)

# 3. Proceed with model fitting if preprocessing was successful
if data_processed is not None and not data_processed.empty and feature_cols:
    print(f"\nProceeding with {len(feature_cols)} features.")
    all_oos_predictions = []
    all_oos_true_values = []
    all_is_predictions = []
    all_is_true_values = []

    test_year_r2_oos = {}
    train_year_r2_is = {}
    components_used = {} # To store number of components used each year

    available_years = sorted(data_processed.index.year.unique())
    actual_test_years = [y for y in available_years if y >= test_start_year and y <= test_end_year]

    if not actual_test_years:
        print(f"\nERROR: No data available in the specified test period ({test_start_year}-{test_end_year}) after preprocessing.")
        pcr_results_all = None
    else:
        print(f"\nStarting PCR annual refitting from {min(actual_test_years)} to {max(actual_test_years)}...")
        print(f"Using Training Start Year: {train_start_year}")

        for current_test_year in actual_test_years:
            loop_start_time = time.time()
            print(f"\n  Processing test year: {current_test_year}")

            current_train_end_year = current_test_year - 1
            if current_train_end_year < train_start_year:
                 print(f"    Skipping year {current_test_year}: Training end year ({current_train_end_year}) is before training start year ({train_start_year}).")
                 continue

            print(f"    Training period: {train_start_year}-{current_train_end_year}")
            train_mask = (data_processed.index.year >= train_start_year) & (data_processed.index.year <= current_train_end_year)
            test_mask = (data_processed.index.year == current_test_year)
            train_df = data_processed.loc[train_mask]
            test_df = data_processed.loc[test_mask]

            if train_df.empty or test_df.empty:
                print(f"    Skipping year {current_test_year}: Not enough data for train ({len(train_df)}) / test ({len(test_df)}) split.")
                continue

            # Check for NaNs before scaling
            if train_df[feature_cols].isna().any().any() or test_df[feature_cols].isna().any().any():
                 print(f"    ERROR: NaNs detected in feature columns for year {current_test_year} before scaling! Skipping.")
                 continue

            # Prepare data
            X_train = train_df[feature_cols].values
            y_train = train_df[target_col].values
            X_test = test_df[feature_cols].values
            y_test = test_df[target_col].values

            # --- Scaling ---
            scaler = StandardScaler()
            try:
                X_train_scaled = scaler.fit_transform(X_train)
                X_test_scaled = scaler.transform(X_test)

                # Check for NaNs again AFTER scaling and impute if necessary
                if np.isnan(X_train_scaled).any() or np.isnan(X_test_scaled).any():
                    print(f"    Warning: NaNs generated during scaling for year {current_test_year}. Imputing with 0.")
                    X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
                    X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

            except ValueError as e:
                print(f"    ERROR during scaling for year {current_test_year}: {e}. Skipping year.")
                continue

            # --- PCA Transformation ---
            try:
                # Fit PCA on the scaled training data to find components
                pca = PCA(n_components=pca_explained_variance_threshold)
                pca.fit(X_train_scaled)
                n_comps = pca.n_components_
                components_used[current_train_end_year] = n_comps
                print(f"    PCA: Selected {n_comps} components to explain {pca_explained_variance_threshold*100:.0f}% variance.")

                # Transform both training and test sets using the fitted PCA
                X_train_pca = pca.transform(X_train_scaled)
                X_test_pca = pca.transform(X_test_scaled)

            except Exception as e:
                print(f"    ERROR during PCA for year {current_test_year}: {e}. Skipping year.")
                continue

            # --- OLS Regression on Principal Components ---
            ols_model = LinearRegression(fit_intercept=True)
            try:
                # Check for NaNs before fitting (should be handled by imputation/PCA)
                if np.isnan(X_train_pca).any() or np.isnan(y_train).any():
                     print(f"    ERROR: NaNs detected in PCA-transformed training data before fitting model for year {current_test_year}. Skipping.")
                     continue
                if np.isnan(X_test_pca).any():
                     print(f"    ERROR: NaNs detected in PCA-transformed test features before prediction for year {current_test_year}. Skipping.")
                     continue

                # Fit OLS on the transformed training data
                ols_model.fit(X_train_pca, y_train)

                # --- In-Sample Prediction (on transformed data) ---
                is_predictions = ols_model.predict(X_train_pca)
                all_is_predictions.extend(is_predictions)
                all_is_true_values.extend(y_train) # Use original y_train
                r2_is_this_year = calculate_r2_is(y_train, is_predictions)
                train_year_r2_is[current_train_end_year] = r2_is_this_year

                # --- Out-of-Sample Prediction (on transformed data) ---
                oos_predictions = ols_model.predict(X_test_pca)
                all_oos_predictions.extend(oos_predictions)
                all_oos_true_values.extend(y_test) # Use original y_test
                r2_oos_this_year = calculate_r2_oos(y_test, oos_predictions)
                test_year_r2_oos[current_test_year] = r2_oos_this_year

                print(f"    Train Year {current_train_end_year} R2_IS : {r2_is_this_year:+.4f}")
                print(f"    Test Year  {current_test_year} R2_OOS: {r2_oos_this_year:+.4f}")
                print(f"    Time for year: {time.time() - loop_start_time:.2f}s")

            except Exception as e:
                print(f"    ERROR during OLS fitting/prediction on PCA components for year {current_test_year}: {e}")

        # --- Overall Results ---
        print(f"\n--- Overall PCR Results (All Features, {pca_explained_variance_threshold*100:.0f}% Var) ---")

        # In-Sample Overall
        overall_r2_is = None
        if all_is_true_values:
            overall_r2_is = calculate_r2_is(np.array(all_is_true_values), np.array(all_is_predictions))
            print(f"\nOverall In-Sample R-squared (R2_IS) across all training periods: {overall_r2_is:.4f}")
            print("\nR2_IS per training period (end year):")
            for year, r2 in sorted(train_year_r2_is.items()):
                print(f"  {train_start_year}-{year}: {r2:.4f} (Components: {components_used.get(year, 'N/A')})")
        else:
            print("\nNo valid in-sample predictions were generated.")

        # Out-of-Sample Overall
        overall_r2_oos = None
        if all_oos_true_values:
            overall_r2_oos = calculate_r2_oos(np.array(all_oos_true_values), np.array(all_oos_predictions))
            print(f"\nOverall Out-of-Sample R-squared (R2_OOS) for period {min(actual_test_years)}-{max(actual_test_years)}: {overall_r2_oos:.4f}")
            print(f"(Paper's PCR (all features) R2_OOS: 0.26%)") # Add PCR comparison [source: 91]
            print("\nR2_OOS per test year:")
            for year, r2 in sorted(test_year_r2_oos.items()):
                 # Find the corresponding training end year to get components used
                 train_end_y = year - 1
                 comps = components_used.get(train_end_y, 'N/A')
                 print(f"  {year}: {r2:.4f} (Used {comps} components from {train_start_year}-{train_end_y} training)")
        else:
            print("\nNo valid out-of-sample predictions were generated for the test period.")

        # Store results
        pcr_results_all = {
            "overall_r2_is": overall_r2_is,
            "overall_r2_oos": overall_r2_oos,
            "yearly_r2_is": train_year_r2_is,
            "yearly_r2_oos": test_year_r2_oos,
            "components_used_per_training_year": components_used,
            "features_used_before_pca": feature_cols
        }

else:
    # This block executes if preprocessing failed
    print("\nData preprocessing failed or resulted in empty DataFrame/features. Cannot proceed with PCR model fitting.")
    pcr_results_all = None # Indicate failure

# Final summary
end_time = time.time()
print(f"\nTotal execution time for PCR (All Features) cell: {end_time - start_time:.2f} seconds")

if 'pcr_results_all' in locals() and pcr_results_all is not None:
    print("\nPCR (All Features) model fitting complete. Results stored in 'pcr_results_all' dictionary.")
    avg_comps = np.mean(list(pcr_results_all['components_used_per_training_year'].values())) if pcr_results_all['components_used_per_training_year'] else 'N/A'
    print(f"Number of features before PCA: {len(pcr_results_all.get('features_used_before_pca', []))}")
    print(f"Average number of PCA components used (explaining {pca_explained_variance_threshold*100:.0f}% variance): {avg_comps:.1f}")
elif 'pcr_results_all' not in locals() or pcr_results_all is None:
     print("\nPCR (All Features) model fitting did not complete successfully.")



#Overall OLS-3 Results (MSE Loss)

In [ ]:
# MSE Loss
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression # Changed from HuberRegressor
from sklearn.metrics import mean_squared_error # Can still be useful for other checks
from sklearn.preprocessing import StandardScaler
import warnings
import time

warnings.filterwarnings("ignore")

# --- Configuration ---
# (Keep the same configuration as Cell 2)
data_folder_path = './'
num_files = 10
file_prefix = 'factor_'
file_suffix = '.csv'

feature_cols = [
    'size_rnk',
    'BPR',
    'mom12'
]
target_col = 'target'
date_col = 'date'
ticker_col = 'ticker'

train_start_year = 2002
# train_end_year = 2015 # This is determined dynamically in the loop
test_start_year = 2019
test_end_year = 2020


# --- Functions ---
# Re-use the preprocessing function from Cell 2 if needed, or assume 'data' is preprocessed
# If 'data' isn't available from Cell 2, you might need to rerun preprocessing or load it
# For simplicity, assume 'data' is available from Cell 2's execution

# --- R-squared Calculation Functions ---
def calculate_r2_oos(y_true, y_pred):
    """
    Calculates Out-of-Sample R-squared using the definition from the paper:
    R2_OOS = 1 - sum( (y_true - y_pred)^2 ) / sum( y_true^2 )
    This benchmarks against a naive forecast of zero.
    """
    numerator = ((y_true - y_pred) ** 2).sum()
    denominator = (y_true ** 2).sum()
    # Handle cases where the sum of squared true values is zero or very close to zero
    if np.isclose(denominator, 0):
        # If numerator is also zero, perfect fit (or all zeros); otherwise, undefined or -inf
        return 1.0 if np.isclose(numerator, 0) else -np.inf
    return 1 - (numerator / denominator)

def calculate_r2_is(y_true, y_pred):
    """
    Calculates In-Sample R-squared using the *same* formula structure as R2_OOS
    for consistency in this specific replication context.
    R2_IS = 1 - sum( (y_train_true - y_train_pred)^2 ) / sum( y_train_true^2 )
    """
    # This uses the same logic as calculate_r2_oos but is applied to training data
    return calculate_r2_oos(y_true, y_pred)

# --- Main Execution ---
print("\n--- Starting OLS-3 Model Fitting (MSE Loss) ---")
start_time = time.time()

# Ensure the preprocessed data DataFrame 'data' exists from the previous cell
if 'data' not in locals() and 'data' not in globals():
     print("ERROR: Preprocessed DataFrame 'data' not found.")
     print("Please ensure the preprocessing step in the previous cell ran successfully.")
     ols_results = None # Indicate failure
elif data is None or data.empty:
    print("ERROR: Preprocessed DataFrame 'data' is None or empty.")
    ols_results = None # Indicate failure
else:
    # --- Proceed with model fitting ---
    all_oos_predictions = []
    all_oos_true_values = []
    all_is_predictions = []  # To store in-sample predictions for overall IS R2
    all_is_true_values = []   # To store in-sample true values for overall IS R2

    test_year_r2_oos = {}     # R2_OOS per test year
    train_year_r2_is = {}    # R2_IS per train period ending year

    available_years = sorted(data.index.year.unique())
    actual_test_years = [y for y in available_years if y >= test_start_year and y <= test_end_year]

    if not actual_test_years:
        print(f"\nERROR: No data available in the specified test period ({test_start_year}-{test_end_year}) after preprocessing. Cannot proceed.")
        ols_results = None
    else:
        print(f"\nStarting OLS-3 annual refitting from {min(actual_test_years)} to {max(actual_test_years)}...")
        print(f"Using Training Start Year: {train_start_year}")

        for current_test_year in actual_test_years:
            loop_start_time = time.time()
            print(f"\n  Processing test year: {current_test_year}")

            current_train_end_year = current_test_year - 1
            if current_train_end_year < train_start_year:
                 print(f"    Skipping year {current_test_year}: Training end year ({current_train_end_year}) is before training start year ({train_start_year}).")
                 continue

            print(f"    Training period: {train_start_year}-{current_train_end_year}")
            train_mask = (data.index.year >= train_start_year) & (data.index.year <= current_train_end_year)
            test_mask = (data.index.year == current_test_year)
            train_df = data.loc[train_mask]
            test_df = data.loc[test_mask]

            if train_df.empty or test_df.empty:
                print(f"    Skipping year {current_test_year}: Not enough data for train ({len(train_df)}) / test ({len(test_df)}) split.")
                continue

            # Final check for NaNs before scaling (should be handled by preprocessing)
            if train_df[feature_cols].isna().any().any() or test_df[feature_cols].isna().any().any():
                 print(f"    ERROR: NaNs detected in features for year {current_test_year} before scaling, after preprocessing! Skipping.")
                 continue

            # Prepare data for scikit-learn
            X_train = train_df[feature_cols].values
            y_train = train_df[target_col].values
            X_test = test_df[feature_cols].values
            y_test = test_df[target_col].values

            # Scale features
            scaler = StandardScaler()
            try:
                X_train_scaled = scaler.fit_transform(X_train)
                X_test_scaled = scaler.transform(X_test)

                # Check for NaNs again AFTER scaling (can happen with constant cols)
                if np.isnan(X_train_scaled).any() or np.isnan(X_test_scaled).any():
                    print(f"    Warning: NaNs generated during scaling for year {current_test_year}. Check for constant columns in training data. Skipping year.")
                    # Impute NaNs caused by scaling constant columns if necessary, or skip
                    # X_train_scaled = np.nan_to_num(X_train_scaled) # Example imputation
                    # X_test_scaled = np.nan_to_num(X_test_scaled)
                    continue # Skip this year if NaNs appear after scaling

            except ValueError as e:
                print(f"    ERROR during scaling for year {current_test_year}: {e}. Skipping year.")
                continue

            # --- OLS Model Training and Prediction ---
            ols_model = LinearRegression(fit_intercept=True) # Use standard OLS

            try:
                # Check for NaNs before fitting/predicting (belt-and-suspenders check)
                if np.isnan(X_train_scaled).any() or np.isnan(y_train).any():
                     print(f"    ERROR: NaNs detected in training data before fitting model for year {current_test_year}. Skipping.")
                     continue
                if np.isnan(X_test_scaled).any():
                     print(f"    ERROR: NaNs detected in test features before prediction for year {current_test_year}. Skipping.")
                     continue

                # Fit the OLS model
                ols_model.fit(X_train_scaled, y_train)

                # --- In-Sample Prediction ---
                is_predictions = ols_model.predict(X_train_scaled)
                all_is_predictions.extend(is_predictions)
                all_is_true_values.extend(y_train)
                r2_is_this_year = calculate_r2_is(y_train, is_predictions)
                train_year_r2_is[current_train_end_year] = r2_is_this_year # Store IS R2 by training end year

                # --- Out-of-Sample Prediction ---
                oos_predictions = ols_model.predict(X_test_scaled)
                all_oos_predictions.extend(oos_predictions)
                all_oos_true_values.extend(y_test)
                r2_oos_this_year = calculate_r2_oos(y_test, oos_predictions)
                test_year_r2_oos[current_test_year] = r2_oos_this_year # Store OOS R2 by test year

                print(f"    Train Year {current_train_end_year} R2_IS : {r2_is_this_year:+.4f}")
                print(f"    Test Year  {current_test_year} R2_OOS: {r2_oos_this_year:+.4f}")
                print(f"    Time for year: {time.time() - loop_start_time:.2f}s")

            except Exception as e:
                print(f"    ERROR during model fitting/prediction for year {current_test_year}: {e}")


        # --- Overall Results ---
        print("\n--- Overall OLS-3 Results (MSE Loss) ---")

        # In-Sample Overall
        if all_is_true_values:
            overall_r2_is = calculate_r2_is(np.array(all_is_true_values), np.array(all_is_predictions))
            print(f"\nOverall In-Sample R-squared (R2_IS) across all training periods: {overall_r2_is:.4f}")
            print("\nR2_IS per training period (end year):")
            for year, r2 in sorted(train_year_r2_is.items()):
                print(f"  {train_start_year}-{year}: {r2:.4f}")
        else:
            print("\nNo valid in-sample predictions were generated.")

        # Out-of-Sample Overall
        if all_oos_true_values:
            overall_r2_oos = calculate_r2_oos(np.array(all_oos_true_values), np.array(all_oos_predictions))
            print(f"\nOverall Out-of-Sample R-squared (R2_OOS) for period {min(actual_test_years)}-{max(actual_test_years)}: {overall_r2_oos:.4f}")
            # Compare to OLS-3 benchmark from paper if desired
            print(f"(Paper's OLS-3 benchmark R2_OOS: 0.16% - Note: Direct comparison depends on exact data/setup)") #
            print("\nR2_OOS per test year:")
            for year, r2 in sorted(test_year_r2_oos.items()):
                print(f"  {year}: {r2:.4f}")
        else:
            print("\nNo valid out-of-sample predictions were generated for the test period.")

        ols_results = {
            "overall_r2_is": overall_r2_is if all_is_true_values else None,
            "overall_r2_oos": overall_r2_oos if all_oos_true_values else None,
            "yearly_r2_is": train_year_r2_is,
            "yearly_r2_oos": test_year_r2_oos
        }


# Final summary
end_time = time.time()
print(f"\nTotal execution time for OLS-3 cell: {end_time - start_time:.2f} seconds")

if 'ols_results' in locals() and ols_results is not None:
    print("\nOLS-3 model fitting complete. Results stored in 'ols_results' dictionary.")
elif 'ols_results' not in locals() or ols_results is None:
     print("\nOLS-3 model fitting did not complete successfully due to errors.")

#Overall NN3 Results (All Features, Ensemble=5)

In [ ]:
# NN 3
# Import necessary libraries
import pandas as pd
import numpy as np
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split # Needed for validation split if not using model.fit's split
from sklearn.metrics import mean_squared_error
import warnings
import time
import random # For setting random seeds

# Suppress TensorFlow warnings and general warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # Suppress TF info/warning messages
tf.get_logger().setLevel('ERROR')
warnings.filterwarnings("ignore")

# --- Configuration ---
data_folder_path = './'
num_files = 10
file_prefix = 'factor_'
file_suffix = '.csv'

# feature_cols will be determined dynamically
target_col = 'target'
date_col = 'date'
ticker_col = 'ticker'

train_start_year = 2002
test_start_year = 2019
test_end_year = 2020

# --- NN Configuration ---
N_ENSEMBLE = 5 # Number of models in the ensemble
EPOCHS = 100 # Max epochs (early stopping will likely stop sooner)
BATCH_SIZE = 512
VALIDATION_SPLIT_RATIO = 0.15 # Use 15% of training data for validation in Early Stopping
EARLY_STOPPING_PATIENCE = 10
L1_REG = 1e-5 # L1 regularization factor (tune if necessary)

# --- Functions ---

# Use the same robust preprocessing function
def preprocess_combined_data(df, date_col, ticker_col, target_col):
    """
    Preprocesses the combined dataframe:
    - Sets date index.
    - Converts potential feature columns to numeric (coercing errors to NaN).
    - Handles infinite values.
    - Drops rows with missing target values.
    - Imputes missing feature values using monthly median, then 0.
    - Returns the processed DataFrame and the list of identified feature columns.
    """
    print("--- Starting Data Preprocessing ---")
    if df is None:
        print("ERROR: Input DataFrame is None.")
        return None, None
    if not isinstance(df, pd.DataFrame):
        print("ERROR: Input is not a pandas DataFrame.")
        return None, None

    df_processed = df.copy()
    print(f"Input shape: {df_processed.shape}")

    # 1. Convert date column and set index
    if date_col in df_processed.columns:
        print(f"Converting '{date_col}' to datetime...")
        df_processed[date_col] = pd.to_datetime(df_processed[date_col], errors='coerce')
        df_processed = df_processed.dropna(subset=[date_col]) # Drop rows where date conversion failed
        df_processed = df_processed.set_index(date_col)
        print(f"Shape after date processing: {df_processed.shape}")
    elif not isinstance(df_processed.index, pd.DatetimeIndex):
        print(f"ERROR: DataFrame index is not a DatetimeIndex and '{date_col}' column not found.")
        return None, None
    print("Date index set.")

    # 2. Identify potential feature columns (all columns except target and ticker)
    potential_feature_cols = [col for col in df_processed.columns if col not in [target_col, ticker_col]]
    print(f"Identified {len(potential_feature_cols)} potential feature columns.")
    if not potential_feature_cols:
        print("ERROR: No potential feature columns found.")
        return None, None

    # Keep only necessary columns for now (target, ticker, potential features)
    cols_to_keep = [ticker_col, target_col] + potential_feature_cols
    missing_cols = [col for col in cols_to_keep if col not in df_processed.columns]
    if missing_cols:
        print(f"ERROR: Missing required columns after date processing: {missing_cols}")
        return None, None
    df_processed = df_processed[cols_to_keep]
    print(f"Shape after selecting columns: {df_processed.shape}")


    # 3. Force Feature Columns to Numeric (Convert non-numeric to NaN)
    print("Attempting to convert identified feature columns to numeric...")
    coerced_count = 0
    numeric_cols_found = [] # Keep track of columns confirmed numeric
    for col in potential_feature_cols:
         if col in df_processed.columns:
              if not pd.api.types.is_numeric_dtype(df_processed[col]):
                  initial_non_numeric = pd.to_numeric(df_processed[col], errors='coerce').isna().sum() - df_processed[col].isna().sum()
                  if initial_non_numeric > 0:
                       # print(f"  Found {initial_non_numeric} non-numeric value(s) in '{col}'. Coercing to NaN.") # Verbose
                       coerced_count += initial_non_numeric
                  df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')
              # Check again if it's numeric now (or was already)
              if pd.api.types.is_numeric_dtype(df_processed[col]):
                  numeric_cols_found.append(col)
              # else: # Don't warn for columns that couldn't be converted (e.g., pure text)
                   # print(f"  WARNING: Column '{col}' could not be converted to numeric. Excluding.")
         else:
              print(f"Warning: Potential feature column '{col}' not found during numeric conversion phase.")

    potential_feature_cols = numeric_cols_found # Update the list to only include numeric ones
    if not potential_feature_cols:
        print("ERROR: No numeric feature columns identified after conversion attempt.")
        return None, None
    print(f"Confirmed {len(potential_feature_cols)} numeric feature columns.")
    if coerced_count > 0:
         print(f"Coerced a total of {coerced_count} non-numeric entries to NaN across features.")

    # 4. Handle infinite values
    df_processed.replace([np.inf, -np.inf], np.nan, inplace=True)
    print("Handled infinite values.")

    # 5. Handle TARGET Variable Missing Values
    initial_rows = len(df_processed)
    n_target_na_initial = df_processed[target_col].isna().sum()
    if n_target_na_initial > 0:
         print(f"Found {n_target_na_initial} missing values in '{target_col}'. Dropping rows...")
         df_processed.dropna(subset=[target_col], inplace=True)
         rows_after_target_drop = len(df_processed)
         rows_dropped = initial_rows - rows_after_target_drop
         if rows_dropped > 0:
              print(f"Dropped {rows_dropped} rows due to missing '{target_col}'.")
    else:
        print(f"No missing values found in '{target_col}'.")

    print(f"Shape after handling target NaNs: {df_processed.shape}")
    if df_processed.empty:
         print("ERROR: All rows dropped due to missing target or date issues. Cannot proceed.")
         return None, None

    # --- Handle FEATURE Missing Values (Median then 0 Imputation) ---
    print("Applying Median+0 imputation to numeric features...")
    try:
        df_processed['month_period'] = df_processed.index.to_period('M')
    except AttributeError as e:
        print(f"Error creating 'month_period': {e}. Ensure index is DatetimeIndex.")
        return None, None

    imputed_count_median = 0
    imputed_count_zero = 0
    final_feature_cols = []

    for col in potential_feature_cols: # Iterate through confirmed numeric columns
        if col in df_processed.columns:
            n_initial_na = df_processed[col].isna().sum()
            if n_initial_na > 0:
                try:
                    # Calculate medians per group, handling potential all-NaN groups by filling NaN medians with 0
                    medians = df_processed.groupby('month_period')[col].median().fillna(0)
                    # Fill NaNs using transform
                    df_processed[col] = df_processed.groupby('month_period')[col].transform(lambda x: x.fillna(medians.loc[x.name]))
                    imputed_this_median = n_initial_na - df_processed[col].isna().sum() # Count how many were filled by median
                    imputed_count_median += imputed_this_median
                except Exception as e:
                    print(f"    WARNING: Could not impute median for '{col}'. Error: {e}. Filling remaining NaNs with 0.")
                    # If transform fails, ensure fillna(0) still runs
                    df_processed[col].fillna(0, inplace=True)
                    imputed_count_zero += df_processed[col].isna().sum() # Count NAs *before* fillna

                # Impute any remaining NaNs (if median was NaN or transform failed) with 0
                n_remaining_na = df_processed[col].isna().sum()
                if n_remaining_na > 0:
                    df_processed[col].fillna(0, inplace=True)
                    imputed_count_zero += n_remaining_na

            # Add to final list if it's still present and numeric (should be)
            if col in df_processed.columns and pd.api.types.is_numeric_dtype(df_processed[col]):
                final_feature_cols.append(col)
            # else: # Don't warn if excluded due to non-numeric type initially
                 # print(f"  WARNING: Column '{col}' is missing or non-numeric after imputation. Excluding.")
        else:
             print(f"Warning: Numeric feature column '{col}' not found during imputation phase.")

    df_processed.drop(columns=['month_period'], inplace=True)
    print(f"Imputation complete. Imputed ~{imputed_count_median} values with median, {imputed_count_zero} values with 0.")

    if not final_feature_cols:
        print("ERROR: No numeric feature columns remain after imputation.")
        return None, None

    nans_in_features = df_processed[final_feature_cols].isna().sum().sum()
    if nans_in_features > 0:
         print(f"WARNING: {nans_in_features} NaNs still remain in final feature columns after imputation! Check imputation logic.")
         # Consider dropping remaining NaNs or using a more robust imputer like IterativeImputer

    print(f"Preprocessing complete. Final shape: {df_processed.shape}")
    print(f"Using {len(final_feature_cols)} features for modeling.")
    if not df_processed.empty:
        print(f"Final data date range: {df_processed.index.min().strftime('%Y-%m-%d')} to {df_processed.index.max().strftime('%Y-%m-%d')}")

    # Return only the columns needed for modeling
    return df_processed[[ticker_col, target_col] + final_feature_cols], final_feature_cols


# --- R-squared Calculation Functions ---
def calculate_r2_oos(y_true, y_pred):
    """Calculates Out-of-Sample R-squared (vs. zero forecast)."""
    numerator = ((y_true - y_pred) ** 2).sum()
    denominator = (y_true ** 2).sum()
    if np.isclose(denominator, 0):
        return 1.0 if np.isclose(numerator, 0) else -np.inf
    return 1 - (numerator / denominator)

def calculate_r2_is(y_true, y_pred):
    """Calculates In-Sample R-squared (vs. zero forecast)."""
    return calculate_r2_oos(y_true, y_pred)

# --- Build NN3 Model Function ---
def build_nn3_model(input_shape, l1_reg=1e-5):
    """Builds the NN3 Keras model with specified architecture."""
    model = keras.Sequential(
        [
            keras.Input(shape=(input_shape,)),
            # Layer 1
            layers.Dense(32, kernel_regularizer=regularizers.l1(l1_reg)),
            layers.BatchNormalization(),
            layers.ReLU(),
            # Layer 2
            layers.Dense(16, kernel_regularizer=regularizers.l1(l1_reg)),
            layers.BatchNormalization(),
            layers.ReLU(),
            # Layer 3
            layers.Dense(8, kernel_regularizer=regularizers.l1(l1_reg)),
            layers.BatchNormalization(),
            layers.ReLU(),
            # Output Layer
            layers.Dense(1, activation='linear') # Linear activation for regression
        ]
    )
    return model

# --- Set Random Seeds for Reproducibility ---
def set_seeds(seed_value=42):
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    tf.random.set_seed(seed_value)
    # Optional: Configure TensorFlow to be deterministic (might impact performance)
    # tf.config.experimental.enable_op_determinism()

# --- Main Execution ---
print(f"\n--- Starting NN3 Model Fitting (All Features, Ensemble={N_ENSEMBLE}) ---")
start_time = time.time()

# 1. Load or ensure 'combined_data' exists
if 'combined_data' not in locals() and 'combined_data' not in globals():
     print("ERROR: 'combined_data' DataFrame not found. Please run the data combination cell first.")
     data_processed = None
     feature_cols = None
else:
    # 2. Preprocess the data AND get the list of features used
    data_processed, feature_cols = preprocess_combined_data(combined_data, date_col, ticker_col, target_col)

# 3. Proceed with model fitting if preprocessing was successful
if data_processed is not None and not data_processed.empty and feature_cols:
    print(f"\nProceeding with {len(feature_cols)} features.")
    num_features = len(feature_cols)

    all_oos_predictions_ensemble_avg = []
    all_oos_true_values = [] # True values are the same for all ensemble members
    all_is_predictions_ensemble_avg = []
    all_is_true_values = [] # True values are the same for all ensemble members

    test_year_r2_oos = {}
    train_year_r2_is = {}
    epochs_stopped_at = {} # Track when early stopping occurred

    available_years = sorted(data_processed.index.year.unique())
    actual_test_years = [y for y in available_years if y >= test_start_year and y <= test_end_year]

    if not actual_test_years:
        print(f"\nERROR: No data available in the specified test period ({test_start_year}-{test_end_year}) after preprocessing.")
        nn3_results_all = None
    else:
        print(f"\nStarting NN3 annual refitting from {min(actual_test_years)} to {max(actual_test_years)}...")
        print(f"Using Training Start Year: {train_start_year}")

        for current_test_year in actual_test_years:
            loop_start_time = time.time()
            print(f"\n  Processing test year: {current_test_year}")

            current_train_end_year = current_test_year - 1
            if current_train_end_year < train_start_year:
                 print(f"    Skipping year {current_test_year}: Training end year ({current_train_end_year}) is before training start year ({train_start_year}).")
                 continue

            print(f"    Training period: {train_start_year}-{current_train_end_year}")
            train_mask = (data_processed.index.year >= train_start_year) & (data_processed.index.year <= current_train_end_year)
            test_mask = (data_processed.index.year == current_test_year)
            train_df = data_processed.loc[train_mask]
            test_df = data_processed.loc[test_mask]

            if train_df.empty or test_df.empty:
                print(f"    Skipping year {current_test_year}: Not enough data for train ({len(train_df)}) / test ({len(test_df)}) split.")
                continue

            # Check for NaNs before scaling
            if train_df[feature_cols].isna().any().any() or test_df[feature_cols].isna().any().any():
                 print(f"    ERROR: NaNs detected in feature columns for year {current_test_year} before scaling! Skipping.")
                 continue

            # Prepare data
            X_train_full = train_df[feature_cols].values
            y_train_full = train_df[target_col].values
            X_test = test_df[feature_cols].values
            y_test = test_df[target_col].values

            # --- Scaling ---
            # Scale based on the FULL training data for this period
            scaler = StandardScaler()
            try:
                X_train_full_scaled = scaler.fit_transform(X_train_full)
                X_test_scaled = scaler.transform(X_test) # Use the same scaler

                # Check for NaNs again AFTER scaling and impute if necessary
                if np.isnan(X_train_full_scaled).any() or np.isnan(X_test_scaled).any():
                    print(f"    Warning: NaNs generated during scaling for year {current_test_year}. Imputing with 0.")
                    X_train_full_scaled = np.nan_to_num(X_train_full_scaled, nan=0.0, posinf=0.0, neginf=0.0)
                    X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

            except ValueError as e:
                print(f"    ERROR during scaling for year {current_test_year}: {e}. Skipping year.")
                continue

            # --- Ensemble Training and Prediction ---
            ensemble_is_predictions = []
            ensemble_oos_predictions = []
            ensemble_epochs_stopped = []

            print(f"    Starting ensemble training (N={N_ENSEMBLE})...")
            for i in range(N_ENSEMBLE):
                print(f"      Training model {i+1}/{N_ENSEMBLE}...")
                # Set different seeds for each model instance
                set_seeds(seed_value=42 + i)

                # Build and compile a new model instance
                model = build_nn3_model(input_shape=num_features, l1_reg=L1_REG)
                model.compile(optimizer=keras.optimizers.Adam(), loss='mean_squared_error')

                # Define Early Stopping callback
                early_stopping = EarlyStopping(
                    monitor='val_loss',       # Monitor loss on validation data
                    patience=EARLY_STOPPING_PATIENCE,         # Number of epochs with no improvement to wait
                    restore_best_weights=True # Restore model weights from the epoch with the best val_loss
                )

                # Train the model using validation_split within fit
                try:
                     history = model.fit(
                         X_train_full_scaled,
                         y_train_full,
                         epochs=EPOCHS,
                         batch_size=BATCH_SIZE,
                         validation_split=VALIDATION_SPLIT_RATIO, # Use Keras' built-in split
                         callbacks=[early_stopping],
                         verbose=0 # Set to 1 or 2 for progress, 0 for silent
                     )
                     stopped_epoch = early_stopping.stopped_epoch
                     if stopped_epoch > 0: # stopped_epoch is 0 if it didn't stop early
                          print(f"        Model {i+1} stopped early at epoch {stopped_epoch + 1}")
                          ensemble_epochs_stopped.append(stopped_epoch + 1)
                     else:
                          print(f"        Model {i+1} completed all {EPOCHS} epochs.")
                          ensemble_epochs_stopped.append(EPOCHS)


                     # Predict (In-Sample and Out-of-Sample)
                     is_preds = model.predict(X_train_full_scaled, batch_size=BATCH_SIZE).flatten()
                     oos_preds = model.predict(X_test_scaled, batch_size=BATCH_SIZE).flatten()

                     ensemble_is_predictions.append(is_preds)
                     ensemble_oos_predictions.append(oos_preds)

                except Exception as e:
                    print(f"      ERROR training/predicting model {i+1} for year {current_test_year}: {e}")
                    # If one model fails, maybe skip it or handle appropriately
                    # For now, we'll just print the error and potentially have fewer models in the average
                    ensemble_epochs_stopped.append(np.nan) # Mark failure
                    continue # Skip to the next ensemble member

                # Clear session to free memory (optional, but can help with many models/features)
                tf.keras.backend.clear_session()


            # --- Aggregate Ensemble Predictions ---
            if not ensemble_oos_predictions: # Check if any models trained successfully
                 print(f"    ERROR: No models in the ensemble trained successfully for year {current_test_year}. Skipping.")
                 continue

            # Average the predictions from all successful models in the ensemble
            avg_is_predictions = np.mean(np.array(ensemble_is_predictions), axis=0)
            avg_oos_predictions = np.mean(np.array(ensemble_oos_predictions), axis=0)

            # Store the averaged predictions and the true values
            all_is_predictions_ensemble_avg.extend(avg_is_predictions)
            all_is_true_values.extend(y_train_full) # True values only need to be stored once per year
            all_oos_predictions_ensemble_avg.extend(avg_oos_predictions)
            all_oos_true_values.extend(y_test) # True values only need to be stored once per year

            # Calculate R-squared based on averaged predictions
            r2_is_this_year = calculate_r2_is(y_train_full, avg_is_predictions)
            r2_oos_this_year = calculate_r2_oos(y_test, avg_oos_predictions)

            train_year_r2_is[current_train_end_year] = r2_is_this_year
            test_year_r2_oos[current_test_year] = r2_oos_this_year
            epochs_stopped_at[current_train_end_year] = np.nanmean(ensemble_epochs_stopped) # Average epochs stopped

            print(f"    Train Year {current_train_end_year} Avg R2_IS : {r2_is_this_year:+.4f}")
            print(f"    Test Year  {current_test_year} Avg R2_OOS: {r2_oos_this_year:+.4f}")
            print(f"    Avg Epochs Stopped: {epochs_stopped_at[current_train_end_year]:.1f}")
            print(f"    Time for year: {time.time() - loop_start_time:.2f}s")


        # --- Overall Results ---
        print(f"\n--- Overall NN3 Results (All Features, Ensemble={N_ENSEMBLE}) ---")

        # In-Sample Overall
        overall_r2_is = None
        if all_is_true_values:
            overall_r2_is = calculate_r2_is(np.array(all_is_true_values), np.array(all_is_predictions_ensemble_avg))
            print(f"\nOverall In-Sample R-squared (R2_IS) across all training periods: {overall_r2_is:.4f}")
            print("\nR2_IS per training period (end year):")
            for year, r2 in sorted(train_year_r2_is.items()):
                print(f"  {train_start_year}-{year}: {r2:.4f} (Avg Epochs: {epochs_stopped_at.get(year, 'N/A'):.0f})")
        else:
            print("\nNo valid in-sample predictions were generated.")

        # Out-of-Sample Overall
        overall_r2_oos = None
        if all_oos_true_values:
            overall_r2_oos = calculate_r2_oos(np.array(all_oos_true_values), np.array(all_oos_predictions_ensemble_avg))
            print(f"\nOverall Out-of-Sample R-squared (R2_OOS) for period {min(actual_test_years)}-{max(actual_test_years)}: {overall_r2_oos:.4f}")
            print(f"(Paper's NN3 (all features) R2_OOS: 0.40%)") # Add NN3 comparison [source: 91]
            print("\nR2_OOS per test year:")
            for year, r2 in sorted(test_year_r2_oos.items()):
                 print(f"  {year}: {r2:.4f}")
        else:
            print("\nNo valid out-of-sample predictions were generated for the test period.")

        # Store results
        nn3_results_all = {
            "overall_r2_is": overall_r2_is,
            "overall_r2_oos": overall_r2_oos,
            "yearly_r2_is": train_year_r2_is,
            "yearly_r2_oos": test_year_r2_oos,
            "avg_epochs_stopped_per_training_year": epochs_stopped_at,
            "features_used": feature_cols,
            "ensemble_size": N_ENSEMBLE
        }

else:
    # This block executes if preprocessing failed
    print("\nData preprocessing failed or resulted in empty DataFrame/features. Cannot proceed with NN3 model fitting.")
    nn3_results_all = None # Indicate failure

# Final summary
end_time = time.time()
print(f"\nTotal execution time for NN3 (All Features, Ensemble) cell: {end_time - start_time:.2f} seconds")

if 'nn3_results_all' in locals() and nn3_results_all is not None:
    print("\nNN3 (All Features, Ensemble) model fitting complete. Results stored in 'nn3_results_all' dictionary.")
    print(f"Number of features used: {len(nn3_results_all.get('features_used', []))}")
    print(f"Ensemble size: {nn3_results_all.get('ensemble_size', 'N/A')}")
elif 'nn3_results_all' not in locals() or nn3_results_all is None:
     print("\nNN3 (All Features, Ensemble) model fitting did not complete successfully.")



#NN3 Results (3 Features, Ensemble=5)

In [ ]:
# 3 feature NN
import pandas as pd
import numpy as np
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import regularizers
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split # Needed for validation split if not using model.fit's split
from sklearn.metrics import mean_squared_error
import warnings
import time
import random # For setting random seeds

# Suppress TensorFlow warnings and general warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # Suppress TF info/warning messages
tf.get_logger().setLevel('ERROR')
warnings.filterwarnings("ignore")

# --- Configuration ---
data_folder_path = './'
num_files = 10
file_prefix = 'factor_'
file_suffix = '.csv'

# Define the specific 3 features to use
specific_feature_cols = ['size_rnk', 'BPR', 'mom12']
target_col = 'target'
date_col = 'date'
ticker_col = 'ticker'

train_start_year = 2002
test_start_year = 2019
test_end_year = 2020

# --- NN Configuration ---
N_ENSEMBLE = 5 # Number of models in the ensemble
EPOCHS = 100 # Max epochs (early stopping will likely stop sooner)
BATCH_SIZE = 512
VALIDATION_SPLIT_RATIO = 0.15 # Use 15% of training data for validation in Early Stopping
EARLY_STOPPING_PATIENCE = 10
L1_REG = 1e-5 # L1 regularization factor (tune if necessary)

# --- Functions ---

# Use the same robust preprocessing function, but we will select columns *after*
def preprocess_combined_data(df, date_col, ticker_col, target_col):
    """
    Preprocesses the combined dataframe:
    - Sets date index.
    - Converts potential feature columns to numeric (coercing errors to NaN).
    - Handles infinite values.
    - Drops rows with missing target values.
    - Imputes missing feature values using monthly median, then 0.
    - Returns the processed DataFrame and the list of ALL identified numeric feature columns.
      (Column selection happens *after* this function call in this script)
    """
    print("--- Starting Data Preprocessing ---")
    if df is None:
        print("ERROR: Input DataFrame is None.")
        return None, None
    if not isinstance(df, pd.DataFrame):
        print("ERROR: Input is not a pandas DataFrame.")
        return None, None

    df_processed = df.copy()
    print(f"Input shape: {df_processed.shape}")

    # 1. Convert date column and set index
    if date_col in df_processed.columns:
        print(f"Converting '{date_col}' to datetime...")
        df_processed[date_col] = pd.to_datetime(df_processed[date_col], errors='coerce')
        df_processed = df_processed.dropna(subset=[date_col]) # Drop rows where date conversion failed
        df_processed = df_processed.set_index(date_col)
        print(f"Shape after date processing: {df_processed.shape}")
    elif not isinstance(df_processed.index, pd.DatetimeIndex):
        print(f"ERROR: DataFrame index is not a DatetimeIndex and '{date_col}' column not found.")
        return None, None
    print("Date index set.")

    # 2. Identify potential feature columns (all columns except target and ticker)
    potential_feature_cols = [col for col in df_processed.columns if col not in [target_col, ticker_col]]
    print(f"Identified {len(potential_feature_cols)} potential feature columns.")
    if not potential_feature_cols:
        print("ERROR: No potential feature columns found.")
        return None, None

    # Keep only necessary columns for now (target, ticker, potential features)
    # Ensure target and ticker are present before selecting
    required_base_cols = [ticker_col, target_col]
    if not all(col in df_processed.columns for col in required_base_cols):
         print(f"ERROR: Missing base columns ({ticker_col}, {target_col}) in DataFrame.")
         return None, None

    cols_to_keep = required_base_cols + potential_feature_cols
    # Ensure no duplicates if ticker/target were somehow in potential_feature_cols
    cols_to_keep = list(dict.fromkeys(cols_to_keep))
    # Check if all columns actually exist
    missing_cols = [col for col in cols_to_keep if col not in df_processed.columns]
    if missing_cols:
        print(f"ERROR: Missing required/potential columns after date processing: {missing_cols}")
        # Remove missing potential features instead of failing?
        potential_feature_cols = [col for col in potential_feature_cols if col in df_processed.columns]
        cols_to_keep = required_base_cols + potential_feature_cols
        print(f"Proceeding with available columns: {cols_to_keep}")
        # return None, None # Or adjust cols_to_keep

    df_processed = df_processed[cols_to_keep]
    print(f"Shape after selecting columns: {df_processed.shape}")


    # 3. Force Feature Columns to Numeric (Convert non-numeric to NaN)
    print("Attempting to convert identified feature columns to numeric...")
    coerced_count = 0
    numeric_cols_found = [] # Keep track of columns confirmed numeric
    for col in potential_feature_cols:
         if col in df_processed.columns:
              if not pd.api.types.is_numeric_dtype(df_processed[col]):
                  initial_non_numeric = pd.to_numeric(df_processed[col], errors='coerce').isna().sum() - df_processed[col].isna().sum()
                  if initial_non_numeric > 0:
                       # print(f"  Found {initial_non_numeric} non-numeric value(s) in '{col}'. Coercing to NaN.") # Verbose
                       coerced_count += initial_non_numeric
                  df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')
              # Check again if it's numeric now (or was already)
              if pd.api.types.is_numeric_dtype(df_processed[col]):
                  numeric_cols_found.append(col)
         # No warning needed if column not found here, handled above
    potential_feature_cols = numeric_cols_found # Update the list to only include numeric ones

    if not potential_feature_cols:
        print("ERROR: No numeric feature columns identified after conversion attempt.")
        return None, None
    print(f"Confirmed {len(potential_feature_cols)} numeric feature columns.")
    if coerced_count > 0:
         print(f"Coerced a total of {coerced_count} non-numeric entries to NaN across features.")

    # 4. Handle infinite values
    df_processed.replace([np.inf, -np.inf], np.nan, inplace=True)
    print("Handled infinite values.")

    # 5. Handle TARGET Variable Missing Values
    initial_rows = len(df_processed)
    n_target_na_initial = df_processed[target_col].isna().sum()
    if n_target_na_initial > 0:
         print(f"Found {n_target_na_initial} missing values in '{target_col}'. Dropping rows...")
         df_processed.dropna(subset=[target_col], inplace=True)
         rows_after_target_drop = len(df_processed)
         rows_dropped = initial_rows - rows_after_target_drop
         if rows_dropped > 0:
              print(f"Dropped {rows_dropped} rows due to missing '{target_col}'.")
    else:
        print(f"No missing values found in '{target_col}'.")

    print(f"Shape after handling target NaNs: {df_processed.shape}")
    if df_processed.empty:
         print("ERROR: All rows dropped due to missing target or date issues. Cannot proceed.")
         return None, None

    # --- Handle FEATURE Missing Values (Median then 0 Imputation) ---
    print("Applying Median+0 imputation to numeric features...")
    try:
        df_processed['month_period'] = df_processed.index.to_period('M')
    except AttributeError as e:
        print(f"Error creating 'month_period': {e}. Ensure index is DatetimeIndex.")
        return None, None

    imputed_count_median = 0
    imputed_count_zero = 0
    final_numeric_cols = [] # Renamed for clarity

    for col in potential_feature_cols: # Iterate through confirmed numeric columns
        if col in df_processed.columns:
            n_initial_na = df_processed[col].isna().sum()
            if n_initial_na > 0:
                try:
                    medians = df_processed.groupby('month_period')[col].median().fillna(0)
                    df_processed[col] = df_processed.groupby('month_period')[col].transform(lambda x: x.fillna(medians.loc[x.name]))
                    imputed_this_median = n_initial_na - df_processed[col].isna().sum()
                    imputed_count_median += imputed_this_median
                except Exception as e:
                    print(f"    WARNING: Could not impute median for '{col}'. Error: {e}. Filling remaining NaNs with 0.")
                    df_processed[col].fillna(0, inplace=True)
                    imputed_count_zero += df_processed[col].isna().sum() # Count NAs *before* fillna

                n_remaining_na = df_processed[col].isna().sum()
                if n_remaining_na > 0:
                    df_processed[col].fillna(0, inplace=True)
                    imputed_count_zero += n_remaining_na

            if col in df_processed.columns and pd.api.types.is_numeric_dtype(df_processed[col]):
                final_numeric_cols.append(col) # Keep track of all numeric cols processed

    df_processed.drop(columns=['month_period'], inplace=True)
    print(f"Imputation complete. Imputed ~{imputed_count_median} values with median, {imputed_count_zero} values with 0.")

    if not final_numeric_cols:
        print("ERROR: No numeric feature columns remain after imputation.")
        return None, None

    nans_in_features = df_processed[final_numeric_cols].isna().sum().sum()
    if nans_in_features > 0:
         print(f"WARNING: {nans_in_features} NaNs still remain in final numeric columns after imputation!")

    print(f"Preprocessing complete. Final shape: {df_processed.shape}")
    print(f"Total {len(final_numeric_cols)} numeric features processed.")
    if not df_processed.empty:
        print(f"Final data date range: {df_processed.index.min().strftime('%Y-%m-%d')} to {df_processed.index.max().strftime('%Y-%m-%d')}")

    # Return the dataframe containing ALL processed numeric columns and the list of their names
    return df_processed, final_numeric_cols


# --- R-squared Calculation Functions ---
def calculate_r2_oos(y_true, y_pred):
    """Calculates Out-of-Sample R-squared (vs. zero forecast)."""
    numerator = ((y_true - y_pred) ** 2).sum()
    denominator = (y_true ** 2).sum()
    if np.isclose(denominator, 0):
        return 1.0 if np.isclose(numerator, 0) else -np.inf
    return 1 - (numerator / denominator)

def calculate_r2_is(y_true, y_pred):
    """Calculates In-Sample R-squared (vs. zero forecast)."""
    return calculate_r2_oos(y_true, y_pred)

# --- Build NN3 Model Function ---
# Model structure remains the same, input shape will change
def build_nn3_model(input_shape, l1_reg=1e-5):
    """Builds the NN3 Keras model with specified architecture."""
    model = keras.Sequential(
        [
            keras.Input(shape=(input_shape,)), # input_shape will be 3 here
            # Layer 1
            layers.Dense(32, kernel_regularizer=regularizers.l1(l1_reg)),
            layers.BatchNormalization(),
            layers.ReLU(),
            # Layer 2
            layers.Dense(16, kernel_regularizer=regularizers.l1(l1_reg)),
            layers.BatchNormalization(),
            layers.ReLU(),
            # Layer 3
            layers.Dense(8, kernel_regularizer=regularizers.l1(l1_reg)),
            layers.BatchNormalization(),
            layers.ReLU(),
            # Output Layer
            layers.Dense(1, activation='linear') # Linear activation for regression
        ]
    )
    return model

# --- Set Random Seeds for Reproducibility ---
def set_seeds(seed_value=42):
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    tf.random.set_seed(seed_value)
    # tf.config.experimental.enable_op_determinism() # Optional for determinism

# --- Main Execution ---
print(f"\n--- Starting NN3 Model Fitting (3 Features: {specific_feature_cols}, Ensemble={N_ENSEMBLE}) ---")
start_time = time.time()

# 1. Load or ensure 'combined_data' exists
if 'combined_data' not in locals() and 'combined_data' not in globals():
     print("ERROR: 'combined_data' DataFrame not found. Please run the data combination cell first.")
     data_processed = None
     all_numeric_features = None
else:
    # 2. Preprocess the data (handles imputation etc. for ALL numeric columns)
    data_processed, all_numeric_features = preprocess_combined_data(combined_data, date_col, ticker_col, target_col)

# 3. Proceed with model fitting ONLY IF preprocessing was successful AND the 3 specific columns exist
feature_cols = specific_feature_cols # Explicitly set the features to use
num_features = len(feature_cols)

if data_processed is not None and not data_processed.empty and all(col in data_processed.columns for col in feature_cols):
    print(f"\nProceeding with {num_features} specific features: {feature_cols}")

    all_oos_predictions_ensemble_avg = []
    all_oos_true_values = []
    all_is_predictions_ensemble_avg = []
    all_is_true_values = []

    test_year_r2_oos = {}
    train_year_r2_is = {}
    epochs_stopped_at = {}

    available_years = sorted(data_processed.index.year.unique())
    actual_test_years = [y for y in available_years if y >= test_start_year and y <= test_end_year]

    if not actual_test_years:
        print(f"\nERROR: No data available in the specified test period ({test_start_year}-{test_end_year}) after preprocessing.")
        nn3_results_3f = None
    else:
        print(f"\nStarting NN3 (3 Features) annual refitting from {min(actual_test_years)} to {max(actual_test_years)}...")
        print(f"Using Training Start Year: {train_start_year}")

        for current_test_year in actual_test_years:
            loop_start_time = time.time()
            print(f"\n  Processing test year: {current_test_year}")

            current_train_end_year = current_test_year - 1
            if current_train_end_year < train_start_year:
                 print(f"    Skipping year {current_test_year}: Training end year ({current_train_end_year}) is before training start year ({train_start_year}).")
                 continue

            print(f"    Training period: {train_start_year}-{current_train_end_year}")
            train_mask = (data_processed.index.year >= train_start_year) & (data_processed.index.year <= current_train_end_year)
            test_mask = (data_processed.index.year == current_test_year)
            # Select only the required columns (target + 3 features) for this period
            train_df = data_processed.loc[train_mask, [target_col] + feature_cols]
            test_df = data_processed.loc[test_mask, [target_col] + feature_cols]

            if train_df.empty or test_df.empty:
                print(f"    Skipping year {current_test_year}: Not enough data for train ({len(train_df)}) / test ({len(test_df)}) split.")
                continue

            # Check for NaNs in the 3 specific features before scaling
            if train_df[feature_cols].isna().any().any() or test_df[feature_cols].isna().any().any():
                 print(f"    ERROR: NaNs detected in the 3 specific feature columns for year {current_test_year} before scaling! Skipping.")
                 continue

            # Prepare data using the 3 features
            X_train_full = train_df[feature_cols].values
            y_train_full = train_df[target_col].values
            X_test = test_df[feature_cols].values
            y_test = test_df[target_col].values

            # --- Scaling ---
            scaler = StandardScaler()
            try:
                X_train_full_scaled = scaler.fit_transform(X_train_full)
                X_test_scaled = scaler.transform(X_test)

                if np.isnan(X_train_full_scaled).any() or np.isnan(X_test_scaled).any():
                    print(f"    Warning: NaNs generated during scaling for year {current_test_year}. Imputing with 0.")
                    X_train_full_scaled = np.nan_to_num(X_train_full_scaled, nan=0.0, posinf=0.0, neginf=0.0)
                    X_test_scaled = np.nan_to_num(X_test_scaled, nan=0.0, posinf=0.0, neginf=0.0)

            except ValueError as e:
                print(f"    ERROR during scaling for year {current_test_year}: {e}. Skipping year.")
                continue

            # --- Ensemble Training and Prediction ---
            ensemble_is_predictions = []
            ensemble_oos_predictions = []
            ensemble_epochs_stopped = []

            print(f"    Starting ensemble training (N={N_ENSEMBLE})...")
            for i in range(N_ENSEMBLE):
                print(f"      Training model {i+1}/{N_ENSEMBLE}...")
                set_seeds(seed_value=42 + i)

                # Build model with input_shape=3
                model = build_nn3_model(input_shape=num_features, l1_reg=L1_REG)
                model.compile(optimizer=keras.optimizers.Adam(), loss='mean_squared_error')

                early_stopping = EarlyStopping(monitor='val_loss', patience=EARLY_STOPPING_PATIENCE, restore_best_weights=True)

                try:
                     history = model.fit(
                         X_train_full_scaled, y_train_full,
                         epochs=EPOCHS, batch_size=BATCH_SIZE,
                         validation_split=VALIDATION_SPLIT_RATIO,
                         callbacks=[early_stopping], verbose=0
                     )
                     stopped_epoch = early_stopping.stopped_epoch
                     if stopped_epoch > 0:
                          print(f"        Model {i+1} stopped early at epoch {stopped_epoch + 1}")
                          ensemble_epochs_stopped.append(stopped_epoch + 1)
                     else:
                          print(f"        Model {i+1} completed all {EPOCHS} epochs.")
                          ensemble_epochs_stopped.append(EPOCHS)

                     is_preds = model.predict(X_train_full_scaled, batch_size=BATCH_SIZE).flatten()
                     oos_preds = model.predict(X_test_scaled, batch_size=BATCH_SIZE).flatten()
                     ensemble_is_predictions.append(is_preds)
                     ensemble_oos_predictions.append(oos_preds)

                except Exception as e:
                    print(f"      ERROR training/predicting model {i+1} for year {current_test_year}: {e}")
                    ensemble_epochs_stopped.append(np.nan)
                    continue

                tf.keras.backend.clear_session()

            # --- Aggregate Ensemble Predictions ---
            if not ensemble_oos_predictions:
                 print(f"    ERROR: No models trained successfully for year {current_test_year}. Skipping.")
                 continue

            avg_is_predictions = np.mean(np.array(ensemble_is_predictions), axis=0)
            avg_oos_predictions = np.mean(np.array(ensemble_oos_predictions), axis=0)

            all_is_predictions_ensemble_avg.extend(avg_is_predictions)
            all_is_true_values.extend(y_train_full)
            all_oos_predictions_ensemble_avg.extend(avg_oos_predictions)
            all_oos_true_values.extend(y_test)

            r2_is_this_year = calculate_r2_is(y_train_full, avg_is_predictions)
            r2_oos_this_year = calculate_r2_oos(y_test, avg_oos_predictions)

            train_year_r2_is[current_train_end_year] = r2_is_this_year
            test_year_r2_oos[current_test_year] = r2_oos_this_year
            epochs_stopped_at[current_train_end_year] = np.nanmean(ensemble_epochs_stopped)

            print(f"    Train Year {current_train_end_year} Avg R2_IS : {r2_is_this_year:+.4f}")
            print(f"    Test Year  {current_test_year} Avg R2_OOS: {r2_oos_this_year:+.4f}")
            print(f"    Avg Epochs Stopped: {epochs_stopped_at[current_train_end_year]:.1f}")
            print(f"    Time for year: {time.time() - loop_start_time:.2f}s")

        # --- Overall Results ---
        print(f"\n--- Overall NN3 Results (3 Features, Ensemble={N_ENSEMBLE}) ---")

        # In-Sample Overall
        overall_r2_is = None
        if all_is_true_values:
            overall_r2_is = calculate_r2_is(np.array(all_is_true_values), np.array(all_is_predictions_ensemble_avg))
            print(f"\nOverall In-Sample R-squared (R2_IS) across all training periods: {overall_r2_is:.4f}")
            print("\nR2_IS per training period (end year):")
            for year, r2 in sorted(train_year_r2_is.items()):
                print(f"  {train_start_year}-{year}: {r2:.4f} (Avg Epochs: {epochs_stopped_at.get(year, 'N/A'):.0f})")
        else:
            print("\nNo valid in-sample predictions were generated.")

        # Out-of-Sample Overall
        overall_r2_oos = None
        if all_oos_true_values:
            overall_r2_oos = calculate_r2_oos(np.array(all_oos_true_values), np.array(all_oos_predictions_ensemble_avg))
            print(f"\nOverall Out-of-Sample R-squared (R2_OOS) for period {min(actual_test_years)}-{max(actual_test_years)}: {overall_r2_oos:.4f}")
            print(f"(Paper's OLS-3 benchmark R2_OOS: 0.16%)") # Compare to OLS-3
            print("\nR2_OOS per test year:")
            for year, r2 in sorted(test_year_r2_oos.items()):
                 print(f"  {year}: {r2:.4f}")
        else:
            print("\nNo valid out-of-sample predictions were generated for the test period.")

        # Store results
        nn3_results_3f = {
            "overall_r2_is": overall_r2_is,
            "overall_r2_oos": overall_r2_oos,
            "yearly_r2_is": train_year_r2_is,
            "yearly_r2_oos": test_year_r2_oos,
            "avg_epochs_stopped_per_training_year": epochs_stopped_at,
            "features_used": feature_cols,
            "ensemble_size": N_ENSEMBLE
        }

elif data_processed is None or data_processed.empty:
    print("\nData preprocessing failed or resulted in empty DataFrame. Cannot proceed.")
    nn3_results_3f = None
else: # Preprocessing succeeded, but the required 3 columns were missing
    missing_req_cols = [col for col in specific_feature_cols if col not in data_processed.columns]
    print(f"\nERROR: The required feature columns {missing_req_cols} were not found in the preprocessed data. Cannot proceed.")
    nn3_results_3f = None


# Final summary
end_time = time.time()
print(f"\nTotal execution time for NN3 (3 Features, Ensemble) cell: {end_time - start_time:.2f} seconds")

if 'nn3_results_3f' in locals() and nn3_results_3f is not None:
    print("\nNN3 (3 Features, Ensemble) model fitting complete. Results stored in 'nn3_results_3f' dictionary.")
    print(f"Features used: {nn3_results_3f.get('features_used', [])}")
    print(f"Ensemble size: {nn3_results_3f.get('ensemble_size', 'N/A')}")
elif 'nn3_results_3f' not in locals() or nn3_results_3f is None:
     print("\nNN3 (3 Features, Ensemble) model fitting did not complete successfully.")

